# 01/08 优化：梯度下降家族 — 学习 Notebook

**学习状态：进行中（2026-11-04），正式测验尚未开始。**

官方英文原文：[08-optimization/docs/en.md](../../../../../../phases/01-math-foundations/08-optimization/docs/en.md)
课程原始代码：[08-optimization/code](../../../../../../phases/01-math-foundations/08-optimization/code/)

本课沿用上一课的学习方式：学生独立作答、自己编写代码；导师逐题验收，实际执行数值测试，并保留原始答案与订正过程。

**官方课程目标**：从零实现 Gradient Descent、SGD with Momentum、Adam；使用 Rosenbrock 函数比较收敛；掌握 mini-batch、学习率调度、凸与非凸问题和鞍点。

**课程节奏**：问题与概念 → Vanilla GD → Momentum → Adam → Rosenbrock 数值实验 → PyTorch 对照 → 四项官方练习 → 三道正式后测。原始 quiz.json 中还有两道前测；正式后测前不披露正确选项。

## 第 1 讲：训练为何需要优化器？

给定参数 $w$ 和损失函数 $L(w)$，目标是：

$$\min_w L(w).$$

在可微位置，梯度 $\nabla L(w)$ 指向局部一阶增长最快的方向。最基本的**梯度下降**使用更新：

$$\boxed{w_{t+1}=w_t-\eta\nabla L(w_t)},\quad\eta>0.$$

负号保证沿负梯度方向迈步，$\eta$ 称为学习率。对于固定方向 $d=-\nabla L(w)$，有方向导数 $\nabla L(w)^\top d=-\|\nabla L(w)\|^2\le0$；但**有限步长的损失并非总会下降**，这要取决于函数局部曲率与学习率。

从本课开始，每种优化器都要区分：更新公式、使用哪些历史状态、每步计算成本、收敛速度和稳定性。用相同初始点与可比较的预算做实验，不直接把某次运行的现象概括为普遍结论。

## 第 1 讲 · 第一题：手算一次梯度下降（等待学生回答）

这是理解检查，**不是**官方测验。

给定：

$$L(w)=(w-3)^2,\quad w_0=5,\quad\eta=0.1.$$

请依次计算：

1. 初始梯度 $L'(w_0)$；
2. 更新后的参数 $w_1=w_0-\eta L'(w_0)$；
3. 更新前的损失 $L(w_0)$ 与更新后的损失 $L(w_1)$，判断是否下降；
4. 用一句话说明为什么更新公式要减去梯度，而不是加上梯度。

导师在学生回答后给出逐项验收与可运行代码，然后进入本课原始前测及 Rosenbrock 函数的梯度推导。

In [ ]:
# 01/08：代码实验从下次学生提交开始。
# 学生先手算一轮梯度下降；教师将逐项验收。


## 后续官方 Exercises 与 Post Quiz（未开始）

官方四道练习（依据英文原文，不提前填入结果）：

1. **Learning rate sweep**：Rosenbrock 上 Vanilla GD 的学习率取 0.0001、0.0005、0.001、0.005、0.01，运行 5000 步并记录最终 loss。
2. **Momentum comparison**：动量分别取 0、0.5、0.9、0.99，追踪完整 loss 曲线、收敛及振荡。
3. **Saddle point escape**：在 $f(x,y)=x^2-y^2$、初始点 $(0.01,0.01)$ 上对比 GD、Momentum、Adam。
4. **Exponential learning-rate decay**：给 GD 加入 $\eta_t=\eta_0\cdot0.999^t$，与固定学习率的 Rosenbrock 结果比较。

官方 post quiz：3 题，完成上面实验后逐题进行并登记到仓库根目录 LEARNING.md。

## 第 1 讲：第一轮梯度下降手算（学生作答与批改）

原始回答：

> 1. 梯度是4 2.w_1是4.6 3.L(w_0)=4, L(w_1) = 2.56

**数值 3/3 正确。**

设 $L(w)=(w-3)^2$，从 $w_0=5$、$\eta=0.1$ 开始：

$$L'(w)=2(w-3),\qquad L'(5)=4.$$
$$w_1=5-0.1\cdot4=4.6.$$
$$L(w_0)=4,\qquad L(w_1)=2.56.$$

损失从 4 降至 2.56。原题还要求说明**为什么沿负梯度方向而非正梯度方向**，学生本轮没有回答；该概念仍需后续理解检查，不算已验收。

第 1 讲后续：在 $L'(w_0)=4>0$ 的位置，向较小 $w$ 方向移动会使局部损失下降；一般情况下，梯度和负梯度的方向导数相反。对于较大有限学习率，并不保证损失每次都下降。

## 课程资料同步说明

学习者希望将上游项目中文译本放到个人仓库对应镜像目录，并从远端拉取至本地 WSL。

本课中文机器译本已经从上游 `translations` 分支复制到：

    learning-artifacts/cn/phases/01-math-foundations/08-optimization/docs/zh.md

**质量标记：未经逐段人工校对的中文机器译文**，上游有明显技术术语误译。本课以官方英文 docs/en.md 为准进行数学与代码校核。

学生此前已克隆 fork 到 WSL，分支是 my-progress。建议先检查本地工作区，再执行：

    cd ~/ai-engineering-from-scratch
    git status --short
    git fetch origin
    git switch my-progress
    git pull --ff-only origin my-progress

若本地 my-progress 不存在，可 `git switch --track origin/my-progress`。本次同步与前序对话中 GitHub 操作都在 fork 的 my-progress 分支，不影响 main。

继续教学时请学生补答：为什么梯度下降沿负梯度方向，而不是正梯度方向？

## 本课课程资源完整镜像与持久规则（已验收）

用户明确要求：每次进入新课，先把官方整个课程目录的**全部内容**镜像到对应中文学习目录，并把此规则写入下次能够读取的持久化项目说明，避免只复制中文讲义。

已在 `my-progress` 分支完成的文件清单：

- 原始官方文件 5/5：`docs/en.md`、`code/optimizers.py`、`code/main.jl`、`quiz.json`、`outputs/prompt-optimizer-guide.md`；均复用官方 Git blob，路径与 SHA 完全一致。
- 中文与个人文件：`docs/zh.md`（上游机器译本，质量待校对）、`quiz.zh.json`（题序、阶段和答案下标与官方一致）、`outputs/prompt-optimizer-guide.zh.md`、`code/learn.ipynb`（保留原有内容）。
- 补充入口：本课 `README.md`、`source-manifest.json`。

已写入仓库持久规则：根目录 `AGENTS.md` 指向 `learning-artifacts/cn/AGENTS.md`，该文件详细规定整课镜像、中文参考、SHA 比对、保护原始学生解答、逐题验收及完成条件；根目录 `LEARNING.md` 和中文目录 `README.md` 也有入口。

已读取 API 目录清单验证全部官方原始文件的 Git blob SHA；已核对中文版 quiz 的 5 道题在 `stage` 与 `correct` 字段上与原版完全一致。Notebook 保持原先 7 个单元后新增本条记录。

后续教学尚待学生补答：为什么梯度下降沿负梯度方向？本课尚未正式完成。

## 第 2 讲：为什么沿负梯度方向，以及 Rosenbrock 函数

第一讲学生已独立算得 $L(w)=(w-3)^2$ 在 $w_0=5$、$\eta=0.1$ 时梯度 4、更新后 $w_1=4.6$、损失由 4 降至 2.56。

一阶泰勒展开：

$$L(\boldsymbol\theta+\Delta\boldsymbol\theta)\approx L(\boldsymbol\theta)+\nabla L(\boldsymbol\theta)^\top\Delta\boldsymbol\theta.$$

对于 $\Delta\boldsymbol\theta=-\eta\nabla L(\boldsymbol\theta)$，

$$L(\boldsymbol\theta-\eta\nabla L)\approx L(\boldsymbol\theta)-\eta\|\nabla L\|_2^2.$$

负梯度方向的方向导数为负（当梯度非零）；足够小的步长可局部降低可微函数损失，但有限步长过大时不能保证下降。

本课权威英文源文件中的测试函数是：

$$f(x,y)=(1-x)^2+100(y-x^2)^2.$$

全局最小值为 $f(1,1)=0$，其附近的狭窄曲线谷地会给优化器带来迂回和振荡问题。

### 学生下一题（待回答）

独立推导 $\partial f/\partial x$ 和 $\partial f/\partial y$，说明对 x 求偏导时如何使用链式法则。**不要提前填写标准导数答案**。学生提交后与解析梯度及有限差分测试交叉验证，之后再让学生自己实现 `rosenbrock_gradient(params)`。

本课知识和代码以后均对照同目录的原始 `docs/en.md`、`code/optimizers.py` 和官方中文学习材料；继续遵循 `learning-artifacts/cn/AGENTS.md` 中完整镜像与逐题验收规则。

## 01/08 · Rosenbrock 偏导验收

学生独立提交：

\[
\frac{\partial f}{\partial x}=-2(1-x)+200(y-x^2)(-2x),\qquad
\frac{\partial f}{\partial y}=200(y-x^2).
\]

**判定：两个偏导均正确。** 化简后：

\[
\nabla f(x,y)=
\begin{pmatrix}
-2(1-x)-400x(y-x^2)\\
200(y-x^2)
\end{pmatrix}.
\]

已在独立 Python 环境实际运行中心差分检查，6 个测试点乘以 2 个偏导共 **12/12 通过**，点为 (-1,1)、(1,1)、(0,0)、(1.4,2.2)、(-0.3,0.8)、(2,-3)，差分步长 1e-6。下个代码单元提供复现用的检查，未把教师程序冒充学生代码。

## 本课中文教材精译说明

已完整重译同课文档 zh.md，保留与英文版一致的 24 个标题、17 个代码及图示块和 4 个外部参考网址。对原文容易造成误解的优化理论陈述以译注注明适用条件。旧的低质量机器译本已替换，英文原文与个人学习记录未覆盖。

**下一步待学生完成：** 自主将偏导表达式写成 Python 函数 rosenbrock_gradient(params)，返回两个偏导组成的列表；教师随后进行实际运行与数值差分验收。

In [ ]:
import math
def f(x, y):
    return (1-x)**2 + 100*(y-x*x)**2
def grad(x, y):
    return (-2*(1-x)+200*(y-x*x)*(-2*x), 200*(y-x*x))
points = [(-1,1),(1,1),(0,0),(1.4,2.2),(-0.3,0.8),(2,-3)]
h = 1e-6
checks = 0
for x,y in points:
    gx,gy = grad(x,y)
    dx = (f(x+h,y)-f(x-h,y))/(2*h)
    dy = (f(x,y+h)-f(x,y-h))/(2*h)
    assert math.isclose(gx,dx,rel_tol=3e-8,abs_tol=4e-7)
    assert math.isclose(gy,dy,rel_tol=3e-8,abs_tol=4e-7)
    checks += 2
print(f'Central-difference gradient checks: {checks}/12 passed')

## 学生独立实现：rosenbrock_gradient(params)（通过）

学生原始代码见下一单元。该实现把此前的解析偏导正确转换为 Python，返回 `[df_dx, df_dy]`，`params` 的两个坐标解包顺序也正确。

在 Python 中已实际执行 14/14 项测试：六个点各两项中心有限差分共 12 项，另检验两个解析点 `grad([1,1]) == [0,0]` 及 `grad([-1,1]) == [-4,0]`。中心差分步长 `1e-6`；判定使用 `math.isclose(rel_tol=3e-8, abs_tol=4e-7)`。

**结论：该函数通过本轮验收。** 不需修改学生代码。

## docs/zh.md 格式修复

检查出中文精译版曾使用 `\[` / `\]` 块级公式定界符、五个无语言标签的代码围栏，以及一处仅供课程网站使用的 `figure` 标识。

已改为 GitHub 推荐的独立 `$$` 数学块、增加 `text` 代码标签，且把 `figure` 专用标签换成可阅读的文本与兼容说明。原文 17 个围栏块的**主体内容**保持一致，24 个标题保持一致，英文源文件不变。

**下一步**：根据官方课程 Build It / Step 2，自主实现 `GradientDescent.step(params, grads)`；确认数学方向和输入输出后再引入动量。

In [ ]:
def rosenbrock_gradient(params):
    # params = [x, y]
    # 返回 [df_dx, df_dy]
    x,y = params
    df_dx =  -2 * (1 - x) - 400 * x * (y - x ** 2)
    df_dy = 200 * ( y - x ** 2)

    return [df_dx, df_dy]

In [ ]:
import math

def rosenbrock_value(point):
    x, y = point
    return (1-x)**2 + 100*(y-x*x)**2

check_points = [(-1,1),(1,1),(0,0),(1.4,2.2),(-0.3,0.8),(2,-3)]
h = 1e-6
checks = 0
for x,y in check_points:
    gx, gy = rosenbrock_gradient([x,y])
    dx = (rosenbrock_value((x+h,y))-rosenbrock_value((x-h,y)))/(2*h)
    dy = (rosenbrock_value((x,y+h))-rosenbrock_value((x,y-h)))/(2*h)
    assert math.isclose(gx,dx,rel_tol=3e-8,abs_tol=4e-7), (x,y,'dx',gx,dx)
    assert math.isclose(gy,dy,rel_tol=3e-8,abs_tol=4e-7), (x,y,'dy',gy,dy)
    checks += 2
assert rosenbrock_gradient([1,1]) == [0,0]
assert rosenbrock_gradient([-1,1]) == [-4,0]
checks += 2
assert checks == 14
print('Rosenbrock gradient: 14/14 PASS')

## 下一项独立编码练习：GradientDescent（等待提交）

公式：`new_p = p - lr*g`（逐坐标计算）。

请只写如下类的 `__init__` 与 `step`，不要先看官方答案：

    class GradientDescent:
        def __init__(self, lr=0.001):
            pass

        def step(self, params, grads):
            pass

`params` 和 `grads` 是长度相同的序列，`step` 返回更新后的参数列表，不需要计算梯度。提交后通过多个维度、正负梯度和不同学习率进行测试。

In [ ]:
class GradientDescent:
        def __init__(self, lr=0.001):
            self.lr = lr

        def step(self, params, grads):
            new_p = params - self.lr * grads
            return new_p

## GradientDescent：学生首次代码提交与测试结果（需订正）

学生独立代码见上方单元；已完整保留原始写法。

**验收：`__init__` 正确；`step` 在要求的 Python 列表输入下失败，暂未通过。**

数学更新公式正确，但代码 `new_p = params - self.lr * grads` 是对 `list` 应用数组向量化运算。

导师实际运行输入 `params=[5.0,3.0]`, `grads=[4.0,-2.0]`, `lr=0.1`：

```text
TypeError: can't multiply sequence by non-int of type 'float'
```

同样的代码若输入为 `numpy.ndarray`，独立实际运行输出 `[4.6 3.2]`，与数学预期一致；但这不是本课程要求的 list 输入。

**待学生独立修复：** 使用 `zip(params, grads)` 逐坐标读取 `p,g`，计算 `p-self.lr*g` 并返回新列表；不提前提供最终完整类答案。接下来检验二维与多维输入、正负梯度、不同学习率。

该错误和订正要求必须保留，不因最后修正成功而删除原始版本。

In [ ]:
class GradientDescent:
        def __init__(self, lr=0.001):
            self.lr = lr

        def step(self, params, grads):
            new_p = [param - self.lr * grad for param, grad in zip(params, grads)]
            return new_p

## GradientDescent 订正完成：通过

学生修订版代码保存在上方独立代码单元。修正了第一次使用 Python list 直接进行向量乘减的错误，改为利用 `zip` 配对参数与梯度，以列表推导式逐坐标更新。

**判定：正确，通过官方 Build It 第 2 步。** 计算公式：

$$\theta_{i,t+1}=\theta_{i,t}-\eta g_{i,t}.$$

本轮实际在 Python 中执行 **6/6 组测试通过**：二维正负混合梯度、默认学习率的三维更新、一维更新、零梯度、多维负参数、tuple 输入返回列表。使用 `math.isclose` 比较浮点数，容差均为 1e-12。

备注：Python `zip` 默认按较短序列截断；本课给定参数与梯度等长的输入契约。生产实现可以增加长度检验，以免悄悄丢失参数。

此前失败的首次提交仍保留在前序单元，未覆盖。

In [ ]:
import math

gd_cases = [
    ('2d-mixed', 0.1, [5.0,3.0], [4.0,-2.0], [4.6,3.2]),
    ('default-3d', 0.001, [1.0,2.0,3.0], [2.0,0.0,-2.0], [0.998,2.0,3.002]),
    ('1d', 0.5, [2.0], [-3.0], [3.5]),
    ('zero-gradient', 0.3, [0.0,10.0,-3.0], [0.0,0.0,0.0], [0.0,10.0,-3.0]),
    ('negative-params', 0.2, [-10.0,5.0,0.5], [5.0,1.0,-0.5], [-11.0,4.8,0.6]),
    ('tuple', 0.1, (5.0,3.0), (4.0,-2.0), [4.6,3.2]),
]
for label, lr, p, g, expected in gd_cases:
    got = GradientDescent(lr).step(p, g)
    assert isinstance(got,list) and len(got)==len(expected), label
    assert all(math.isclose(a,b,rel_tol=1e-12,abs_tol=1e-12) for a,b in zip(got,expected)), label
print('GradientDescent correction: 6/6 PASS')

## 官方 Build It 第 3 步：Momentum（待学生）

本课官方的状态更新为：

$$v_t=\beta v_{t-1}+g_t,\qquad\theta_{t+1}=\theta_t-\eta v_t.$$

在第一次调用 `step` 时，把 `velocity` 初始化为与 `params` 同维度的全零列表；随后每次 `step` 都更新并保留 `velocity`。`beta` 即动量系数。

**下一项理解检查：** 假设一维参数 $w_0=5$，$v_0=0$，$\eta=0.1$，$\beta=0.9$，连续两步收到相同梯度 $g_1=g_2=4$。手算第一步、第二步的 $(v,w)$，并指出它与两步普通梯度下降的差别。学生回答后再进入 Momentum 类的代码实现。

保持逐题推进，不提前向学生给出这道手算题的数值答案。

## Momentum 手算验收

学生原始答案：

> v_1 =  4, w_1 = 4.6, v_2 = 7.6,w_2 = 3.84

**4/4 正确。**

按本课动量更新公式：

$$v_t=\beta v_{t-1}+g_t,\quad w_t=w_{t-1}-\eta v_t,$$

代入 $w_0=5$, $v_0=0$, $\beta=0.9$, $\eta=0.1$, $g_1=g_2=4$：

$$v_1=4,\quad w_1=4.6,\quad v_2=0.9\cdot4+4=7.6,\quad w_2=4.6-0.76=3.84.$$

普通 GD 两步后得到 $w_2=4.2$；动量法第二步使用历史梯度，单步更新幅度 0.76，大于普通 GD 的 0.4。

这是额外的理解检查，非官方 Exercises 或 Quiz。

## 官方流程审计：补全此前遗漏（2026-11-06）

学生要求严格遵循课程。现核对本课官方 docs/en.md、quiz.json 及仓库 AGENTS.md 与个人学习规约。

**目前不算完全严格：** 原始 5 个文件已完整镜像；Build It 第 1 步 Rosenbrock 和第 2 步 GradientDescent 已完成并验收；Build It 第 3 步 Momentum 只完成手算，还未完成类实现。

**明确遗漏：** 原版 quiz.json 的两道 stage=pre 在教学开始前没有出题。现在只能按“迟做的前测题”补做，不能当成真实课前知识基线。

**尚未完成：** Build It 第 4 步 Adam、第 5 步完整对比；Use It、Ship It；四道官方 Exercises；三道 stage=post 正式后测。

此前手算、数值差分和附加测试是有价值的教学补充，但不能替代官方练习。整课仍未完成，不更新 LEARNING.md 完成状态。

接下来先补做原始两道前测（按官方题号，每次一题，清楚标明补做），然后恢复 Momentum 类实现。

## 官方前测第 1/2 题（迟做，等待学生回答）

本题原来自 quiz.json 中的第 1 道 stage=pre，因此前教学已开始，这次只作补题，不作为真正的学习前诊断成绩。

**训练神经网络时，“优化”（optimization）具体意味着什么？**

A. 寻找使损失函数最小的模型参数

B. 使程序代码运行得更快

C. 减少模型参数的数量

D. 选择更好的训练硬件

请学习者先给出选项与理由。评分后再呈现第二道前测，未答之前不写出正确选项。

## 01/08 · 原本的前测第 1 题补答记录

学生原始回复：

> 01/08 官方补做前测第1题：选择 A。理由：不对,你还漏了讲解的部分,你要先读取它的prompt,然后再慢慢讲解

选项 **A 正确（1/1）**：训练中的优化是寻找使损失函数尽可能小的模型参数。学生的后半段是指出教学流程错误，并不是对选项的概念解释，不应伪称已经给出数学理由。

按 Skill 审计：`skills/learn/SKILL.md` 在 Step 2 要求先讲 Problem/Concept、先暂停理解题，再逐步推导和分段讲代码；Step 3 正式进行的是 `stage=post` 的题目。原版 `quiz.json` 中的 `pre` 题不是该 Skill 的强制门槛。之前“必须完成全部 pre 才能讲下一节”的说法错误，已在中文 AGENTS.md 修正。前测第 2 题不强制补做。

## 本课 Prompt 的实际用途与恢复教学计划

- **教学 Skill**：`skills/learn/SKILL.md`（仓库原始教学规范），适用于所有课程；必须先读取并遵循交互式教学流程。
- **本课英文讲义**：`docs/en.md`（课程数学与代码权威）；其中文镜像为 `docs/zh.md`。
- **Ship It Prompt**：`outputs/prompt-optimizer-guide.md`（为具体 ML 任务选择优化器、学习率和 schedule 的交付提示词），并非授课提示词；到 Ship It 阶段必须讲解与实际使用它。

在本章第 1、2 步 Build It 已完成、Momentum 手算正确的情况下，用户指出解释不足。因此先回到漏讲的 Concept 部分，把**梯度和优化器的区别、learning rate、batch/SGD/mini-batch、为什么动量能减轻狭窄谷地振荡**分小节互动讲透，然后继续 Momentum 类的独立实现。教学过程每次只讲一小节，不一次性塞进所有公式与代码。

这条安排用于纠正教学节奏，不清空任何已提交代码，不将新增理解练习算作官方考试。

## 恢复 `learn` 课程：核心概念理解检查（等待学生）

### 为什么已经知道梯度，还需要优化器？

把梯度理解为当前位置的局部坡度，优化器是决定**如何利用坡度、历史方向、尺度**来更新模型参数的规则。普通 GD 只看当前梯度，Momentum 还会利用过去若干步的历史信息。

在 Rosenbrock 狭窄弯曲的谷地中，**横穿谷地**的梯度方向可能频繁反转，**沿谷地前进**的分量在若干步内却可能保持一致。因此普通 GD 容易左右来回跳，Momentum 会使反转方向的历史项相互抵消、在一致方向上积累速度。这解释了为什么不同方向的优化速度可能不同。

**请先用文字回答：** 如果梯度连续两步在某个方向上正负交替，另一个方向上一直保持同号，Momentum 对这两个方向分别可能产生什么影响？

学生回答后再推导对应分量的更新，接着解释 batch vs mini-batch；此刻不要催促直接写完整 SGDMomentum 类。

## Momentum：历史梯度与各坐标振荡的理解检查（已批改）

学生原始答案：

> momentum会导致 梯度在x方向减左右震荡的频率,会增加y方向的梯度

**方向判断基本正确，但两处表述需订正：**

1. x 方向：动量加权累积可以抑制正负交替梯度带来的更新**幅度**，不能保证改变交替的**频率**。
2. y 方向：动量不会直接增大当前真实梯度 $g_t$；它通过累积相同方向的历史梯度增大速度 $v_t$，从而可能放大参数更新。

本课动量采用 $v_t=\beta v_{t-1}+g_t$，$\theta_t=\theta_{t-1}-\eta v_t$，初始 $v_0=(0,0)$。设 $\beta=0.9$，梯度 $g_1=(4,1)$，$g_2=(-4,1)$，则：

$$v_1=(4,1),\qquad v_2=0.9(4,1)+(-4,1)=(-0.4,1.9).$$

因此第二步原始梯度是 $(-4,1)$，但有效更新向量是 $-\eta(-0.4,1.9)$。展开递推：

$$v_t=\sum_{k=0}^{t-1}\beta^k g_{t-k}\quad(v_0=0).$$

该解释对一致方向的积累与振荡方向的部分抵消给出数学依据，但并非对任意梯度序列均保证抑制振荡或改善收敛。测试单元复算两步速度。

**下一步补充理解题**：从 $v_2=(-0.4,1.9)$ 出发，如果下一步 $g_{3,y}=-1$，且 $\beta=0.9$，预测第 3 步的 $v_{3,y}$ 是正、负还是零？解释为何梯度已经反向，参数更新方向却可能尚未反向。学生答完再进入 Momentum 类实现。

In [ ]:
beta = 0.9
velocity = [0.0, 0.0]
observed = []
for gradients in ([4.0, 1.0], [-4.0, 1.0]):
    velocity = [beta * v + g for v, g in zip(velocity, gradients)]
    observed.append(velocity[:])
assert all(abs(a-b)<1e-12 for got,expected in zip(observed,[[4.0,1.0],[-0.4,1.9]]) for a,b in zip(got,expected))
print('Momentum two-step concept test: PASS', observed)

## Momentum：当前梯度反向而速度保持同向（通过）

学生原始回答：

> v3,y是正数,因为前面速度加新的梯度没有反向

**判定：正确。** 学生理解历史动量会阻止更新方向立即跟随瞬时梯度反转。

根据上一个理解检查 $v_{2,y}=1.9$，$g_{3,y}=-1$，$\beta=0.9$：

$$v_{3,y}=0.9\times1.9-1=0.71>0.$$

若学习率 $\eta>0$，则参数更新 $\Delta y_3=-\eta v_{3,y}=-0.71\eta<0$，即**速度仍为正、参数仍向较小的 y 移动**。当前梯度负号不等于瞬时速度立刻反向；注意梯度与速度是不同变量。

这一题是 Concept 的补充理解检查，不计入官方 Post Quiz 或 Exercises。

## 官方 Build It 第 3 步：Momentum 类——先完成状态初始化

解释：普通 `GradientDescent.step()` 仅保存学习率，Momentum 还需要在**同一个对象**里持续保留历史速度，因此不能每次 `step` 都重新初始化速度。

官方初始化接口：

```python
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        # 保存学习率、动量系数和速度状态
        ...
```

学生本轮只需要独立完成 `__init__`，先不写 `step` 的最终实现。要求将 `lr`、`momentum` 保存为对象属性；`velocity` 初始为 `None`，代表还不知道参数维度、尚未初始化。之后再讨论首次 `step` 如何构造长度相同的零向量和继续更新的区别。

理解问题：**为什么速度不能在每一次 `step()` 开头都设为零？** 学生写好初始化方法并解释后再进入更新逻辑。

后续仍需要完成未充分讲解的 SGD / batch / mini-batch 的 Concept，以及 Adam、LR schedule、凸/非凸、损失地形，再做完整对照实验和官方练习。按官方 `skills/learn/SKILL.md` 概念优先、5～15 行分段互动，不跳过。

In [ ]:
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        # 由你完成
        self.lr = lr
        self.momentum = momentum
        self.velocity = 0

## SGDMomentum 第一版初始化：需要订正

学生原始代码保存在上一单元，聊天附带解释为：

> 因为我们的速度要用前面的速度更新,不能设置为0

**评价**：`lr`、`momentum` 两个属性初始化正确；理解到速度必须保留历史状态。`velocity=0` 是标量整数，和官方类的初始状态约定不一致。

本课官方 `code/optimizers.py` 中的 `SGDMomentum.__init__` 使用 `self.velocity = None`；第一次进入 `step` 时按参数维度初始化零列表，之后复用速度。零向量只应在初次使用时设置，而不是每一步重置。将它初始化为整数 0 会导致 `if self.velocity is None` 无法触发，且后续在 `zip(self.velocity, grads)` 中因整数不可迭代而失败。

**实际 Python 检查结果**：构造 `SGDMomentum()` 得到 `lr=0.001`、`momentum=0.9`、`velocity=0`；`velocity is None` 为 False；以 0 和梯度列表运行 `zip` 抛出 `TypeError: 'int' object is not iterable`。

**订正要求**：只将初始化阶段的速度改成 `None`，说明 `None` 是尚未建立速度列表的状态标识，以及为什么不能在 `__init__` 中根据未知参数维度预先创建适当长度的零列表。先不要求学生完成整个 `step`。

学生在消息中与代码同一行书写的中文是回答解释，不作为可执行的 Python 源码，Notebook 将其保留在此。

## Momentum 延迟初始化理解检查

学生原始回答：因为我们不知道我们要调整的速度的列表的形状,我们后面根据参数的维度创建零向量

**理解正确。** `None` 表示尚未初始化，首次 `step` 根据 `len(params)` 建立同维零向量。此前代码 `self.velocity=0` 尚未由学生订正。

下一步让学生独立完成 `step` 中检测 `self.velocity is None` 并建立零向量的条件分支，之后才写动量递推。

In [ ]:
def step(self, params, grads):
        # 请你完成首次初始化判断
        self.velocity = [0.0] * len(params)

## Momentum step 首次初始化分支——第一次提交（需订正）

学生原始代码保存在上一个代码单元。

**判断：** `[0.0] * len(params)` 构造同维零向量正确，但没有判断 `self.velocity is None`，因此每次调用 `step` 都清除历史速度，破坏 Momentum 的跨步状态。

订正要求：仅在速度尚未初始化（`self.velocity is None`）时创建零列表；否则保留历史速度。先不实现速度递推和参数更新。学生独立订正后再验收。

In [ ]:
def step(self, params, grads):
        # 请你完成首次初始化判断
        if self.velocity == None:
            self.velocity = [0.0] * len(params)

## SGDMomentum `step()` 初始化条件：第二次提交通过

学生原始版本见上一代码单元（本次使用 `== None`，而非官方的 `is None`）。

**本轮局部任务通过。** `if self.velocity == None` 在当前对象状态为 `None` 或普通 Python 列表时可以实现仅初始化一次的效果，之后不会清零。对 `None` 的身份测试推荐写作 `is None`，语义更清楚，且避免数组类型的逐元素比较歧义。

实际 Python 执行 2/2 通过：

1. `velocity=None`，`params=[1.,2.,3.]`，调用后获得 `[0.,0.,0.]`；
2. 人为将 `velocity=[4.,2.,-1.]`，再次调用仍保持该值，不被清零。

注意：**这仅说明初始化分支完成；整个 `step` 尚未完成**，不应把完整的 Momentum 优化器标记为通过。

下一步只实现公式 `v_t=beta*v_{t-1}+g_t` 的逐参数列表更新，复用上节学到的 `zip`。速度更新后再独立实现参数更新。

In [ ]:
class _MomentumInitOnly:
    def __init__(self):
        self.velocity = None
    def step(self, params, grads):
        if self.velocity == None:
            self.velocity = [0.0] * len(params)

instance = _MomentumInitOnly()
instance.step([1.,2.,3.], [1.,1.,1.])
assert instance.velocity == [0.,0.,0.]
instance.velocity = [4.,2.,-1.]
instance.step([1.,2.,3.], [1.,1.,1.])
assert instance.velocity == [4.,2.,-1.]
print("Momentum state initialization: 2/2 PASS")

In [ ]:
def step(self, params, grads):
        # 请你完成首次初始化判断
        if self.velocity == None:
            self.velocity = [0.0] * len(params)
        self.velocity = [self.momentum * v + g for v,g in zip(self.velocity,grads)]
        new_p = [param - self.lr * v for param, v in zip(params, self.velocity)]

        return new_p

## SGDMomentum 完整 step：验收通过，注意构造器前置条件

学生提交的 step 原始内容保存在上一单元，已经正确实现一次性状态初始化、速度递推及参数更新。
实际在 Python 环境运行 13/13 检查通过，涉及两步 1D、三步 2D、正负梯度反转、零动量、tuple 和输入对象不变性。下一个代码单元可复现。

**重要：本次只提交 step 函数。测试假设构造器为之前演示的正确形式 self.velocity = None；此前学生提交的构造器还曾是 self.velocity = 0，尚未提交新的完整构造器。若仍用 0，则旧状态判定不会触发，随后 zip(self.velocity,grads) 抛出 TypeError。** 因此正式判断是 step 已通过，整个类还需确认将构造器改为 None。

建议将 if self.velocity == None 改成 if self.velocity is None（None 单例的标准身份判断）。官方 Rosenbrock 实验的 gradients 是完整解析梯度，不包含随机采样，此处需要与 mini-batch SGD 区分。

In [ ]:
import math
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        self.lr, self.momentum, self.velocity = lr, momentum, None

    def step(self, params, grads):
        if self.velocity == None:
            self.velocity = [0.0] * len(params)
        self.velocity = [self.momentum * v + g for v,g in zip(self.velocity,grads)]
        new_p = [param - self.lr * v for param, v in zip(params, self.velocity)]
        return new_p

results=[]
def check(name,got,expected):
    assert len(got)==len(expected)
    assert all(math.isclose(a,b,rel_tol=1e-12,abs_tol=1e-12) for a,b in zip(got,expected)),(name,got,expected)
    results.append(name)
m=SGDMomentum(.1,.9)
w1=m.step([5.],[4.]); check('1d-first-w',w1,[4.6]); check('1d-first-v',m.velocity,[4.])
w2=m.step(w1,[4.]); check('1d-second-w',w2,[3.84]); check('1d-second-v',m.velocity,[7.6])
m=SGDMomentum(.1,.9)
w1=m.step([1.,2.],[4.,1.]); check('2d-first-w',w1,[.6,1.9]); check('2d-first-v',m.velocity,[4.,1.])
w2=m.step(w1,[-4.,1.]); check('2d-second-w',w2,[.64,1.71]); check('2d-second-v',m.velocity,[-.4,1.9])
w3=m.step(w2,[0.,-1.]); check('2d-third-w',w3,[.676,1.639]); check('2d-third-v',m.velocity,[-.36,.71])
m=SGDMomentum(.2,0); check('zero-momentum',m.step([3.,4.],[-2.,3.]),[3.4,3.4])
m=SGDMomentum(.1,.9); p=[1.,2.]; g=[.2,-.5]; m.step(p,g); assert p==[1.,2.] and g==[.2,-.5]; results.append('non-mutating')
m=SGDMomentum(.1,.9); check('tuple-input',m.step((5.,3.),(4.,-2.)),[4.6,3.2])
assert len(results)==13
print('SGDMomentum: 13/13 PASS')

## 概念理解检查：SGD 与 mini-batch

官方类名是 SGDMomentum，但在 Rosenbrock 实验中提供给 step 的是完整解析梯度；优化器如何更新参数，与调用者如何估计梯度，是两个层面。

问题：如果有 1000 个训练样本，每一步随机取 32 个计算梯度并调用你的 SGDMomentum.step(params,grads)，这属于全批量、单样本 SGD 还是 mini-batch SGD？相比使用全部 1000 个样本，梯度具有什么特点？

按一问一答等待学生回复，随后进入官方 Adam 概念与数学推导。

## 概念理解检查：SGD / Mini-batch（通过）

学生原始回答：

> 这个属于mini-batch sgd 相对于1000个样本点全批量梯度下降这个梯度的特点就是快些,然后噪声是中等,但是我要顺便问一下sgd的s是什么英文

**判定：正确。** 每次随机抽取32/1000样本计算梯度属于 Mini-batch SGD；相较全批量，单次梯度计算通常成本更低，并存在样本抽样噪声。需注意总收敛时间未必更短，噪声大小取决于批大小和样本梯度分布，不能固定称为“中等”。

学生询问的术语：`SGD` = **Stochastic Gradient Descent**；`S` = **Stochastic**，意为随机的。

若完整训练集目标为 $L(\theta)=\frac{1}{1000}\sum_{i=1}^{1000}\ell_i(\theta)$，其梯度为

$$g_{\mathrm{full}}=\frac{1}{1000}\sum_{i=1}^{1000}\nabla\ell_i(\theta).$$

每次均匀抽取大小为32的 batch $B$，

$$g_B=\frac{1}{32}\sum_{i\in B}\nabla\ell_i(\theta),\qquad\mathbb E_B[g_B]=g_{\mathrm{full}}.$$

这里的无偏等式要求均匀抽样及恰当的样本均值估计；不表示单次小批量梯度等于完整梯度。该部分是 Concept 课堂理解检查，不计入官方练习/后测。

### 下一步（等待学生回答）

进入 Adam 的概念：相同学习率下，不同参数的梯度历史尺度可能不同。向学生提出直观问题：如果两个参数的梯度长期分别约为 100 和 0.01，而使用同一个学习率，更新尺度会有什么差异，可能带来什么优化问题？先回答直觉，再推导 Adam 的一阶、二阶矩及偏差修正。

## Adam 自适应学习率动机：更新幅度计算（通过）

学生原始回答：

> 参数a的更新幅度是0.1 参数B的更新幅度是0.00001

**数值计算 2/2 正确。** 固定学习率 $\eta=0.001$，梯度 $g_A=100$、$g_B=0.01$：

$$\Delta\theta_A=-0.001(100)=-0.1,\quad \Delta\theta_B=-0.001(0.01)=-10^{-5}.$$

更新幅度相差 $10^4$ 倍。在相同学习率下，参数的梯度尺度差异会导致参数更新尺度差异，但不能仅凭尺度不同便认定有问题，还要结合曲率与参数尺度。

**该题概念解释部分待答，不能标记整题全部完成。** 下一问：若增大学习率使 B 移动更快，A 有什么潜在风险？若缩小学习率确保 A 稳定，B 又会怎么样？等学生回答后再深入讲解 Adam 的一阶矩、二阶矩和偏差修正。

## Adam 动机：固定学习率两难（理解检查通过）

学生原始回答：

> 如果B更新的更快,那么A的更新幅度也跟着变大,可能会跳过最优点,左右跳动,如果A学习率调小,B就变得更小了,可能需要很多很多step才能缓慢移动

**结论：正确。** 使用相同学习率、梯度分别为 $100$ 与 $0.01$ 时，放大全局学习率会同步放大两个参数的更新，可能令大梯度方向越过低损失区并振荡甚至发散；缩小学习率可能导致小梯度方向移动缓慢。该现象是否发生取决于方向曲率、参数尺度与具体目标，不是必然发生。

学生已完成公式计算 $|\Delta\theta_A|=0.1$、$|\Delta\theta_B|=10^{-5}$ 和含义解释两个部分；本轮理解检查通过。

### Adam 结构初步讲解与下一道理解检查

官方课程提出：Adam 为每个坐标分别维护两个历史统计量：

1. **一阶矩估计** $m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$：带符号的梯度滑动平均，反映历史上的更新方向。
2. **二阶原始矩估计** $v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$：梯度平方的滑动平均，反映历史梯度量级（不是方差）。

它们与 SGDMomentum 课程之前的无归一化速度 $u_t=\beta u_{t-1}+g_t$ 不应混同：Adam 的一阶矩使用 $(1-\beta_1)$ 权重，且随后有偏差修正。

下一问：若某参数的梯度交替为 $+10,-10,+10,-10,\ldots$，为什么直接对梯度取滑动平均可能互相抵消，而对梯度平方取滑动平均不会？只解释直觉；之后再做一阶矩、二阶矩和偏差修正的数值推导。

## Adam：一阶矩、二阶原始矩的含义和区别

学生原始回答及提问：

> 按照我的理解一阶矩,它的梯度是g_t是可以看到正负的,代表着方向,二阶矩的g_t ^ 2 只有大小,代表着梯度的量级,我需要你给我讲一下二阶矩一节矩,

**理解判断：正确。** 一阶矩保留梯度符号，反映梯度的平均方向；二阶原点矩通过平方反映梯度的平方尺度，不受正负号抵消影响。注意它不是平均绝对值，也不是方差。对随机变量 X，

$$\mu_1'=\mathbb E[X],\quad\mu_2'=\mathbb E[X^2],\quad\operatorname{Var}(X)=\mathbb E[X^2]-(\mathbb E[X])^2.$$

例：四个梯度 [10,-10,10,-10]，平均梯度 0，但平均平方梯度 100，方差 100；四个梯度 [10,10,10,10]，平均 10、平均平方 100、方差 0。二阶原点矩无法单独反映梯度波动。

Adam 的逐坐标指数移动平均（设对应坐标的第 t 步梯度为 g_t）：

$$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,$$

$$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.$$

其中 m 是一阶矩估计，v 是二阶原始矩估计；它们是对不断变化的训练梯度的历史加权估计，并非任何时刻都严格等于固定分布下的真期望。常用 \beta_1=0.9, \beta_2=0.999。

初始化 m_0=v_0=0 时，早期 EMA 存在趋近零的初始化偏差。Adam 使用

$$\hat m_t=\frac{m_t}{1-\beta_1^t},\qquad\hat v_t=\frac{v_t}{1-\beta_2^t},$$

$$\theta_t=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$

这里 \sqrt{\hat v_t} 是历史梯度均方根尺度的估计；这一项帮助根据每个坐标的梯度历史尺度归一化更新。偏差修正在梯度同分布且期望固定的理想化条件下可推导期望的初始缩小因子；训练中梯度随参数变化，因而不是对所有情形的精确无偏保证。

**单步示例：** 如果 m_0=v_0=0、g_1=10，则 m_1=1，v_1=0.1，\hat m_1=10，\hat v_1=100。

**下一次交互问题（等待学生回答）：** 为什么第一步 m_1=1、v_1=0.1，而偏差修正后分别成为 10 和 100？请用 EMA 的初始值解释。答完后进入第二步正负交替梯度示例与 Adam 代码分段实现。

## Adam EMA 与偏差修正：概念讲解（2026-11-06）

### 学生原始回答与提问

> 因为是从零初始化的,早期估计偏向零,我需要你给我讲一下为什么要这样修正,什么是估计,为什么叫指数移动平均

**理解检查：通过。** 学生正确理解零初始化引起早期估计趋零，并主动要求解释估计、指数移动平均及修正的来源。

### 1. 统计估计与 Adam 的状态

对于固定分布下随机变量 $X$，未知一阶原点矩 $\mu=\mathbb E[X]$ 可由样本平均 $\hat\mu_n=\frac{1}{n}\sum_i X_i$ 估计；二阶原点矩 $\mathbb E[X^2]$ 可由平方样本平均估计。估计量是样本到数值的函数，估计值是代入具体样本后的结果。

Adam 对每个参数的历史梯度保留两种时变统计量：$m_t$ 跟踪带符号梯度的指数平均，$v_t$ 跟踪平方梯度的指数平均；在非平稳训练过程中，它们不是对一个固定真实期望的保证无偏估计。

### 2. 为什么是指数移动平均

$$m_t=\beta m_{t-1}+(1-\beta)g_t,\quad m_0=0.$

$$m_t=(1-\beta)\sum_{k=1}^{t}\beta^{t-k}g_k.$$

最近的 $g_t$ 权重 $(1-\beta)$，倒数第二个为 $(1-\beta)\beta$，倒数第三个为 $(1-\beta)\beta^2$，依次指数衰减。它随每个新观察值持续更新，因此称为“移动”平均，无需存储全部梯度。

### 3. 从权重之和严格推导偏差修正

$$\sum_{k=1}^t(1-\beta)\beta^{t-k}=(1-\beta)\sum_{j=0}^{t-1}\beta^j=1-\beta^t.$$

由于 $m_0=0$，在早期权重总和小于 1；对恒定梯度 $g_k=c$，有 $m_t=c(1-\beta^t)$，并不是 $c$。因此归一化：

$$\hat m_t=\frac{m_t}{1-\beta_1^t},\qquad \hat v_t=\frac{v_t}{1-\beta_2^t}.$$

更一般地，在各步梯度满足同一个期望 $\mathbb E[g_k]=\mu$ 的条件下，有 $\mathbb E[m_t]=(1-\beta^t)\mu$，故修正后 $\mathbb E[\hat m_t]=\mu$；即便样本非独立，这一线性期望结论仍成立。若梯度期望随训练发生变化，归一化只修正零初始化产生的缩小因子，并不保证对当前梯度期望完全无偏。

### 4. 数字例子

取 $\beta=0.9$，$m_0=0$，每次 $g_k=10$。递推得到 $m_1=1$、$m_2=1.9$、$m_3=2.71$、$m_4=3.439$；除以 $1-0.9^t$ 后四次均为 10。

二阶矩取 $\beta_2=0.999$，若 $g_1=10$，则 $v_1=0.001\times100=0.1$，$\hat v_1=0.1/(1-0.999)=100$。二阶矩不等于方差，其平方根对应梯度均方根尺度。

### 下一道理解检查（等待学习者）

只问一题：取 $\beta=0.8$，初始 $m_0=0$，连续两个梯度 $g_1=g_2=5$。请计算 $m_1$、$m_2$、$\hat m_2=m_2/(1-\beta^2)$，并解释修正后的结果为何是 5。先由学生独立解答，再继续后续 Adam 代码。

## EMA 偏差修正理解检查：学生作答与验收

学生原始回答：

> m1 = 1 m2 = 0.8 + 1 = 1.8 因为 /1-beta^2 相当于对 m_2进行统计意义的无偏修正

**评价：** $m_1=1$、$m_2=1.8$ 两步计算正确；知道除以 $1-\beta^2$ 是对零初始化造成的偏差修正；但没有显式计算题目要求的 $\hat m_2$，由导师补全如下。

$$m_1=0.8\cdot0+0.2\cdot5=1,$$

$$m_2=0.8\cdot1+0.2\cdot5=1.8,$$

$$\hat m_2=\frac{1.8}{1-0.8^2}=\frac{1.8}{0.36}=5.$$

进一步说明为什么修正后得到 5：两步对应的未归一化权重分别为 $(1-\beta)\beta=0.16$、$(1-\beta)=0.2$，相加是 $1-\beta^2=0.36$。于是偏差修正后的权重之和等于 1。对于常数梯度 5，加权平均必然为 5。

**统计条件：** 如果所有历史梯度具有同一均值 $\mu$，则 $\mathbb E[m_t]=(1-\beta^t)\mu$，而 $\mathbb E[\hat m_t]=\mu$。非平稳训练过程不能保证相对当前梯度分布的完全无偏性。

本理解检查：前两步数值正确，修正原理基本正确，最后一步的数值由导师补充；仍归入补充理解练习，不计入官方 Exercises/Post Quiz。

### 下一项一题一问：Adam 实际更新

设第 1 步梯度 $g_1=10$，Adam 的初始 $m_0=v_0=0$，$\beta_1=0.9$、$\beta_2=0.999$；已知 $m_1=1$、$v_1=0.1$、$\hat m_1=10$、$\hat v_1=100$。令 $\eta=0.001$ 且仅为简化计算取 $\epsilon=0$。请学生计算：

$$\Delta\theta_1=-\eta\frac{\hat m_1}{\sqrt{\hat v_1}+\epsilon}.$$

问：第 1 步参数更新量是多少？为什么它不是普通 GD 对 $g_1=10$ 的 $-0.01$？学生答完再进入多步数值与 Adam 代码实现。

## Adam 第一步参数更新计算（理解检查通过）

学生原始回答：

> delta_theta_one = - 0.001 因为我们对这个进行了幅度修正

**判定：数值正确，概念方向正确；进一步明确偏差修正与梯度尺度归一化的区别。**

已知 $\eta=0.001$、$\hat m_1=10$、$\hat v_1=100$、为本题简化设 $\epsilon=0$：

$$\Delta\theta_1=-\eta\frac{\hat m_1}{\sqrt{\hat v_1}}=-0.001\cdot\frac{10}{10}=-0.001.$$

而普通 GD 对 $g_1=10$ 的更新为 $-0.01$。偏差修正是将 EMA 初始化权重归一化（$m/(1-\beta_1^t)$、$v/(1-\beta_2^t)$）；本题改变更新幅度的直接原因是用 $\sqrt{\hat v_t}+\epsilon$ 对平均梯度进行逐坐标尺度归一化。

特殊首步情形：若 $m_0=v_0=0$，$g_1\neq0$ 且 $\epsilon=0$，偏差修正后有 $\hat m_1=g_1$、$\hat v_1=g_1^2$，因此 $\Delta\theta_1=-\eta\operatorname{sign}(g_1)$。此结论不适用于 $g_1=0$ 或 $\epsilon>0$ 的精确等式，也不代表全部 Adam 迭代。

### 官方 Build It 第 4 步：Adam 状态初始化（等待学生实现）

根据官方源码的参数表，构造器接口为：

```python
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        # 学生自己完成
        pass
```

学生本轮仅需实现构造器，保存 lr、beta1、beta2、epsilon，并初始化一阶/二阶矩状态 m、v 为 None，时间步计数 t 为 0。设计动机：Adam 和 Momentum 一样不知道未来 params 长度，要首次 step 才初始化同维零向量；另外 t 记录已做的更新次数，用于 $1-\beta_1^t$ 与 $1-\beta_2^t$。

请学生同时解释：为什么需要成员变量 t，不能每次调用 step 都重新让 t=1？之后逐段实现 moment EMA 与偏差修正。

In [ ]:
class Adam:
    def __init__(
        self,
        lr=0.001,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):
        # 请你自己完成
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0因为我们在偏差修正的时候需要t来修正

## Adam 构造器：学生首次提交与真实运行检查

学生提交代码保留在上方独立代码单元。此处不将粘贴时附在代码末尾的中文解释偷偷删掉。

**代码结构和数学理解正确，但原始源码尚无法直接运行。** 最后一行 `self.t = 0因为我们在偏差修正的时候需要t来修正` 把中文解释粘在表达式后，实际 Python `compile` 报 `SyntaxError: invalid syntax`（第16行）。学生的意思是把 `t` 初始化为 0，供偏差修正时使用。

为了独立验收算法逻辑，仅将该行解释分离为 `self.t = 0` 后，在 Python 中实际构造类并验证状态：默认参数和自定义参数的两项断言 **2/2 PASS**。其余 6 个属性和初始化值均与官方 `code/optimizers.py` 的 Adam 构造器一致。

**状态判断：** 理解和初始化逻辑合格；原始代码待将中文解释独立成一行或注释，不能标为原始源码可运行。`t` 是累计 `step` 更新次数：构造器中为 0，第一次执行 step 时变 1，第二次变 2；不能在每次调用 step 时重置为 1，否则偏差修正的分母 `1-beta1**t` 和 `1-beta2**t` 错误。

### 下一轮：Adam.step 的初始化片段

先让学生将 `self.t=0` 单独放到合法的代码行。之后实现 `step(self,params,grads)` 的第一段：当 `self.m is None` 时根据 `len(params)` 把 `m`、`v` 建成等长零列表；每次调用时 `self.t += 1`。不要提前代写后面的逐参数一阶矩、二阶矩公式；学生独立完成后再验收。

按 `skills/learn/SKILL.md` 每次讲一个短代码片段、理解确认后再继续。

In [ ]:
# 对学生原始代码的检查（已实际在 Python 运行）
raw_student_last_line = 'self.t = 0因为我们在偏差修正的时候需要t来修正'
# 原始完整代码无法编译，报 SyntaxError（第 16 行）。
# 将解释文字分离后，以下断言用于验收逻辑，不代表原始源码通过。
class Adam:
    def __init__(self,lr=0.001,beta1=0.9,beta2=0.999,epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0
assert vars(Adam()) == dict(lr=0.001,beta1=0.9,beta2=0.999,epsilon=1e-8,m=None,v=None,t=0)
assert vars(Adam(lr=.02,beta1=.7,beta2=.88,epsilon=1e-6)) == dict(lr=.02,beta1=.7,beta2=.88,epsilon=1e-6,m=None,v=None,t=0)
print('Cleaned constructor: 2/2 PASS; original raw input: SyntaxError')

In [ ]:
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.m = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

## Adam 的 step 初始化片段：第一次提交（需要订正）

学生原始代码见上方独立单元。代码判断 `self.m is None and self.v is None`，按参数维度建立零列表，以及每次增加 `self.t` 的方向均正确。
**需要修正**：第二条赋值重复写成 `self.m = [0.0] * len(params)`，导致 `self.v` 仍为 `None`。在本课初始 `m=v=None` 且一起初始化的前提下，联合条件能工作，但官方只判断 `m is None`。

已实际在 Python 中运行原始片段，`params=[1.0,2.0]`、`grads=[0.3,-0.1]`；执行后 `m=[0.0,0.0]`，`v=None`，`t=1`。模拟随后二阶矩的 `list(zip(self.v, grads))`，得到 `TypeError: 'NoneType' object is not iterable`。

**本轮验收：尚未通过，等待订正。** 请只将第二次初始化写入 `self.v`，保留其余代码；不提前展开 Adam 一阶矩与二阶矩更新。

In [ ]:
# 实际检验学生本轮 step 初始化片段（原样保留）
class AdamInitCheck:
    def __init__(self):
        self.m = None
        self.v = None
        self.t = 0
    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.m = [0.0] * len(params)
        self.t += 1
a = AdamInitCheck()
a.step([1.0, 2.0], [0.3, -0.1])
assert a.m == [0.0, 0.0]
assert a.v is None
assert a.t == 1
try:
    list(zip(a.v, [0.3, -0.1]))
except TypeError as exc:
    print('Expected failure:', repr(exc))
else:
    raise AssertionError('Expected TypeError from uninitialized v')

In [ ]:
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

## Adam step 初始化订正：通过（6/6 检查）

学生完整原始订正代码见上一代码单元。修复了第一次提交中把第二个赋值误写成 self.m 的问题，现在分别初始化 self.m 和 self.v，并在条件分支外对 self.t 加 1。

**仅此初始化代码片段验收通过，尚未实现 EMA 更新及参数更新。**

实际使用 Python 运行下面的测试（代码保存在下一单元），全部 6/6 通过：

1. 首次调用 m 为零向量；
2. 首次调用 v 为零向量；
3. 首次调用 t=1；
4. 第二次调用保留人为赋值的 m、v 历史状态；
5. 第二次调用 t=2；
6. 单参数 tuple 输入仍创建正确维度的状态。

代码检查：`if self.m is None and self.v is None` 在本课约定下（m、v 总是一同建立）成立；官方写法为 `if self.m is None`，两种写法在正常状态流上等效。`self.t += 1` 放在 if 外，确保每步累加。

In [ ]:
class _AdamStateOnly:
    def __init__(self):
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1

obj = _AdamStateOnly()
obj.step([1., 2., 3.], [.1, .2, .3])
assert obj.m == [0., 0., 0.]
assert obj.v == [0., 0., 0.]
assert obj.t == 1
obj.m = [2., 3., 4.]
obj.v = [5., 6., 7.]
obj.step([1., 2., 3.], [.1, .2, .3])
assert obj.m == [2., 3., 4.] and obj.v == [5., 6., 7.]
assert obj.t == 2
obj2 = _AdamStateOnly()
obj2.step((10.,), (.5,))
assert obj2.m == [0.] and obj2.v == [0.] and obj2.t == 1
print('Adam initialization step: 6/6 PASS')

## Build It 第 4 步：Adam 一阶矩递推（等待学生）

已知 `self.m` 首次调用被初始化为零向量、`self.beta1` 在构造器里保存，当前 `grads` 是各个参数的梯度。

$$m_{t,i}=\beta_1 m_{t-1,i}+(1-\beta_1)g_{t,i}.$$

下一轮只用 `zip(self.m,grads)` 和列表推导式更新 `self.m`，不要提前写二阶矩、偏差修正、参数更新。

先给学生独立尝试机会；检查其代码时注意 `(1-beta1)` 的乘积项不能漏掉。

In [ ]:
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ^ 2 for v,g in zip(self.v, grads)]

## Adam 一阶矩 / 二阶矩递推：学生首次提交与错误分析

**学生原始代码**保存于上一个代码单元。此前只布置一阶矩，学生主动实现了一阶矩和二阶矩。

**判断**：一阶矩递推正确：$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$。二阶矩的结构正确，但 Python `g ^ 2` 不是乘方，`^` 是按位异或；应使用 `g ** 2`。

实际 Python 运行学生代码：构造器状态正确，调用 `step([1.0,2.0],[2.0,-3.0])` 时抛出 `TypeError("unsupported operand type(s) for ^: 'float' and 'int'")`；发生异常前 `t` 已从0变1，`m` 已更新为约 `[0.2,-0.3]`，`v` 保持 `[0.0,0.0]`，说明该异常可能遗留**部分更新状态**，不能当作一次成功的 step。

在仅把 `g ^ 2` 改成 `g ** 2` 后（只作为导师验收，不替代学生订正），输入两步梯度 `[2,-3]` 和 `[-2,3]`、默认 $\beta_1=0.9$、$\beta_2=0.999$，得到：

- 第一步 $m\approx[0.2,-0.3]$，$v\approx[0.004,0.009]$，$t=1$。
- 第二步 $m\approx[-0.02,0.03]$，$v\approx[0.007996,0.017991]$，$t=2$。

上述结果实际在 Python 中执行，并用浮点容差断言检验成功。代码附在下一单元。**原始提交本轮未通过，等待学生纠正运算符后重新提交**。尚未开始偏差修正或参数更新实现。

In [ ]:
import math

class _AdamMomentRaw:
    def __init__(self):
        self.beta1, self.beta2 = 0.9, 0.999
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ^ 2 for v,g in zip(self.v, grads)]

raw = _AdamMomentRaw()
try:
    raw.step([1.0, 2.0], [2.0, -3.0])
except TypeError as exc:
    print('Raw code error:', str(exc))
else:
    raise AssertionError('Expected TypeError due to ^')
assert raw.t == 1 and raw.v == [0.0, 0.0]

class _AdamMomentFixed(_AdamMomentRaw):
    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * (g ** 2) for v,g in zip(self.v, grads)]

fixed = _AdamMomentFixed()
fixed.step([1.0, 2.0], [2.0, -3.0])
assert fixed.t == 1
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.m, [0.2, -0.3]))
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.v, [0.004, 0.009]))
fixed.step([1.0, 2.0], [-2.0, 3.0])
assert fixed.t == 2
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.m, [-0.02, 0.03]))
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.v, [0.007996, 0.017991]))
print('Corrected moment update: two steps passed')

### 下一项只修正一个运算符（等待学生）

请学生把二阶矩更新中的 `g ^ 2` 改为 Python 的乘方写法，保持其余代码，重新提交完整递推片段。验收通过后，再按照官方 Build It 第 4 步进入 `m_hat` / `v_hat` 偏差修正与逐参数更新。

In [ ]:
# 2026-11-06：学生原始订正，Adam.step 的状态、一阶矩与二阶矩片段
# 保留学生原始格式，不代表此片段可作为独立顶层代码直接执行。
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ** 2 for v,g in zip(self.v, grads)]

In [ ]:
# 学生原始偏差修正代码（2026-11-06，正确）
m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]

In [ ]:
# 学生原始参数更新代码（2026-11-06，待订正）
# 这里的列表推导式 for 放错位置，保留作错误记录
new_params = [p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon for p, m_h ,v_h in zip(params,m_hat,v_hat)) ]
return new_params

## Adam 续学：原始答案、诊断与讲评（2026-11-06）

**递推订正：通过。** 学生已把二阶矩乘方由先前的 `g ^ 2` 改为 `g ** 2`，与公式 $v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$ 一致。

**偏差修正：通过。** 学生独立写出：
```python
m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]
```

**参数更新：首次提交未通过。** 学生原始代码在前一代码单元保留。表达式虽然可经 Python 语法解析，但 `for ...` 被写在分母括号内，形成生成器表达式，而 `p` 位于生成器作用域之外；独立运行该表达式触发 `NameError: name 'p' is not defined`。导师给出下列参考订正（**非学生提交**）：
```python
new_params = [
    p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon)
    for p, m_h, v_h in zip(params, m_hat, v_hat)
]
return new_params
```

**学生第一次手算回答（原文）：**
> p_new = 2.99 ,因为我们要的v_hat是二阶矩的修正,所以要加上sqrt

数值正确：$p=3,\eta=0.01,\hat m=2,\hat v=4,\epsilon=10^{-8}$，故 $p_{new}=3-0.01\cdot2/(2+10^{-8})\approx2.99000000005$。概念解释不充分：开平方是将梯度平方的尺度恢复为与梯度相同的量纲，从而逐坐标规范化；它不是因为“修正”这一操作本身需要开平方。

**学生第二次理解检查（原文）：**
> 1.情况A的delta_p_a是-0.01 2.情况B的delta_p_b 是-0.001,因为他们的量纲不一样

两个数值均正确。给定 $\hat m_t=0.5$、$\eta=0.01$、忽略 $\epsilon$：A 的 $\hat v_t=0.25$ 导致 $\Delta p_A=-0.01$；B 的 $\hat v_t=25$ 导致 $\Delta p_B=-0.001$。**原因应为二阶矩的数值 / 历史梯度平方尺度不同，而不是量纲不同**：A 与 B 的 $\hat v$ 都有梯度平方的相同量纲。

**实际运行核验（教师独立在 Python 中完成；后附可复验代码单元）：**原始更新表达式运行得 `NameError`；参考订正构成完整 Adam 后，首步两参数输出约为 `[2.99000000005, 4.00999999995]`，第二步约为 `[2.9816940247702366, 4.01773077269116]`，时间步 `t=2`；A/B 手算断言通过。这些是**教师参考实现的测试**，不代表学生原始完整代码已经独立通过。

**课程交付物提醒：** 本课 `outputs/prompt-optimizer-guide.md` 和中文副本已检查。它们作为本课 Ship It 输出和训练任务选择优化器的实践 Prompt，非官方 `skills/learn/SKILL.md` 的替代教学规范。按 `learning-artifacts/cn/AGENTS.md`，续课应先读取 Skill、英文原文、Notebook 和 Quiz，核对完整镜像及输出文件。

In [ ]:
# 教师参考订正 + 独立回归验证（不会覆盖学生原始代码）
import ast
import math

raw_line = "new_params = [p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon for p, m_h ,v_h in zip(params,m_hat,v_hat)) ]"
ast.parse(raw_line)   # 语法可解析，但变量作用域不正确

class _AdamReferenceCheck:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr, self.beta1, self.beta2, self.epsilon = lr, beta1, beta2, epsilon
        self.m, self.v, self.t = None, None, 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m, g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ** 2 for v, g in zip(self.v, grads)]
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]
        return [
            p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon)
            for p, m_h, v_h in zip(params, m_hat, v_hat)
        ]

ref = _AdamReferenceCheck()
first = ref.step([3.0, 4.0], [2.0, -2.0])
second = ref.step(first, [0.5, -0.3])
assert all(math.isclose(a, b, abs_tol=1e-10) for a, b in zip(first, [2.99, 4.01]))
assert ref.t == 2
assert math.isclose(-0.01 * 0.5 / math.sqrt(0.25), -0.01)
assert math.isclose(-0.01 * 0.5 / math.sqrt(25), -0.001)
print("Teacher reference check passed:", first, second, "t =", ref.t)

### 当前进度与下一步（未完成）

Adam 的 EMA 和偏差修正已经由学生独立提交并通过。参数更新的列表推导式有首次原始提交和教师参考纠错，**待学生确认其订正后的完整 `step()` 并运行真实测试**。随后做 Rosenbrock 优化器对比、与 PyTorch 核验、官方 Exercises 和 Post Quiz。整课仍为**进行中**，根目录 `LEARNING.md` 暂不增加“已完成”记录。

## Adam 完整类：学生原始提交（2026-11-06）

下面完全保留学生原始提交文本，特别是最后列表推导式的多余右括号。由于原始代码不能编译，放在 Markdown 代码围栏中保存，不作为可执行的正确代码。

```python
class Adam:
    def __init__(
        self,
        lr=0.001,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):
        # 请你自己完成
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ** 2 for v,g in zip(self.v, grads)]

        # 3. 偏差修正
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]

        # 4. 更新参数
        new_params = [p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon) for p, m_h ,v_h in zip(params,m_hat,v_hat)) ]

        # 5. 返回更新后的参数
        return new_params
```

## 完整 Adam 实现：语法诊断与 PyTorch 对照

**原始代码验收：语法未通过；核心数学更新逻辑已正确。**

- 学生原始提交的参数更新行：`zip(params,m_hat,v_hat)) ]`，多了一个右括号 `)`；Python 实际 `compile` 报 `SyntaxError: closing parenthesis ')' does not match opening parenthesis '['`，位于原始类代码第 31 行（单独测试的紧凑版本中为第 19 行）。
- 作为独立脚本还需要 `import math`。构造器、零状态初始化、时间步、自适应 EMA、偏差修正与参数更新的数学结构均正确。
- 教师测试时**只修正多余括号**，并在独立模块上下文中导入 `math`，没有改变学生算法。
- 使用 PyTorch `2.10.0+cpu`、`float64`、`lr=0.01`、`betas=(0.9, 0.999)`、`eps=1e-8`、`weight_decay=0`、`foreach=False`。初始参数 `[3.0, 4.0]`，依次输入梯度 `[2.0,-2.0]`、`[0.5,-0.3]`、`[-1.0,0.4]`。三次结果：
  - 第 1 步：`[2.99000000005,4.00999999995]`；
  - 第 2 步：`[2.9816940247702366,4.01773077269116]`；
  - 第 3 步：`[2.978708507348488,4.022352235773342]`。
- 三步分别与 PyTorch 最大绝对误差均为 0（此输入和环境下的实际浮点结果），此外 `t=3`、历史矩状态长度为 2、全零梯度保持参数不变均通过。

**边界限制**：`zip(params, grads)` 在长度不同时静默截断；当前实现遵循官方教学参考，但生产级实现宜验证维度一致。非空参数与有限数值的约束也未做全覆盖测试。

**当前完成程度**：Build It 的 Adam 算法组成已写出，并有一个语法小错误；仅针对修正后的版本完成多步对照验证。原始提交不能记为运行通过；待学生自主提交修正的完整类，再进入 Rosenbrock 学习阶段。课程及官方 Exercises/Post Quiz 均未完成。

In [ ]:
# 教师复现实验：严格区分“原始代码的 SyntaxError”和“只修正括号后的算法验证”
import math
import torch

class _StudentAdamOneParenFixed:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr, self.beta1, self.beta2, self.epsilon = lr, beta1, beta2, epsilon
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m, g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ** 2 for v, g in zip(self.v, grads)]
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]
        new_params = [p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon) for p, m_h, v_h in zip(params, m_hat, v_hat)]
        return new_params

learner = _StudentAdamOneParenFixed(lr=0.01)
params = [3.0, 4.0]
tensor = torch.nn.Parameter(torch.tensor(params, dtype=torch.float64))
optimizer = torch.optim.Adam([tensor], lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0, foreach=False)
for i, g in enumerate([[2.0, -2.0], [0.5, -0.3], [-1.0, 0.4]], start=1):
    params = learner.step(params, g)
    tensor.grad = torch.tensor(g, dtype=torch.float64)
    optimizer.step()
    max_err = max(abs(x - y) for x, y in zip(params, tensor.detach().tolist()))
    assert max_err < 1e-12
    print(f'step {i}: params={params}, max_abs_diff={max_err:.3e}')
assert learner.t == 3 and len(learner.m) == 2 and len(learner.v) == 2
zero = _StudentAdamOneParenFixed(lr=0.01)
assert zero.step([1.0, -2.0], [0.0, 0.0]) == [1.0, -2.0]
print('PASS: 3 steps vs PyTorch, state persistence, zero gradients')

### 互动学习：下一步

先请学生将完整代码中的多余 `) `删掉，并确认 `import math`。核心 Adam 实现即可判定为 Build It 的本段完成。随后转入官方 Rosenbrock 优化器数值实验及学习率对照；学生独立完成，每次只分配一道计算或代码任务，及时记录原始答案、验证结果，最后完成官方 Exercises 和 Post Quiz。

## Build It 第 5 步：Rosenbrock 优化器对比（2026-11-06）

学生原始回复：

> 可以,我已经删除了,继续

这是对上一轮 Adam 类中多余 `) ` 的**自行修正确认**；本轮没有再次粘贴修正后的完整代码。上一轮教师已对仅删去多余括号的实现用 PyTorch 做三步数值对照通过，因此将 Adam 核心算法 Build It 标为通过（保留“用户声明已修正、未再次实际获得修正文件”的证据边界）。

遵循官方 `phases/01-math-foundations/08-optimization/docs/en.md` 的 **Build It / Step 5: Run and compare**。前文已完成 Rosenbrock 解析梯度推导与数值验证，不重复题目。

公式：$f(x,y)=(1-x)^2+100(y-x^2)^2$，从 $(-1,1)$ 开始、目标为 $(1,1)$。在官方演示中以 5000 次更新比较 GD lr=0.0005、SGDMomentum lr=0.0001 momentum=0.9、Adam lr=0.01。由于三种方法 lr 不同，结果仅说明给定设置下的行为；不是“Adam 普遍更快”的严格证明。

本轮只布置 **`optimize` 循环**的一小段：给定 optimizer、损失函数 func、梯度函数 grad_func、初始参数 start 及 steps，学生自己完成梯度计算、step 调用、历史参数的拷贝记录。指出 `func` 保留在官方接口里但该循环内部未使用，用于后续 final loss 比较。

```python
def optimize(optimizer, func, grad_func, start, steps=5000):
    params = list(start)
    history = [params[:]]
    for _ in range(steps):
        grads = ???      # 当前点梯度
        params = ???     # 调用优化器
        history.append(???)  # 保存独立快照
    return history
```

再让学生预测：为什么保存 `params[:]` 比直接保存可变的原列表 `params` 更稳妥？等待独立提交和实际运行测试后，进行三种优化器对比，再进入官方 Exercises、Use It/Ship It、Post Quiz。课程仍未完成，根目录 `LEARNING.md` 暂不增加完成行。

In [ ]:
def optimize(optimizer, func, grad_func, start, steps=5000):
    params = list(start)
    history = [params[:]]

    for _ in range(steps): 
        # 1. 计算当前参数对应的梯度
        grads = grad_func(params)

        # 2. 调用优化器，得到新的参数
        params = optimizer.step(params,grads)

        # 3. 保存这一轮更新后的参数
        history.append(params[:])

    return history

## Rosenbrock 比较实验：`optimize()` 验收（2026-11-06）

**学生原始代码：通过。** 与官方 `Build It / Step 5` 中的 `optimize` 主体一致：先记录初始参数，循环计算梯度并调用 `optimizer.step`，逐步保存参数列表快照，最后返回 `history`。此接口保留了 `func` 参数，但该循环内部并未使用它，损失函数用于之后对历史参数求损失及对比。

**学生原始理解解释：**
> 因为用params[:]是相当于读取params的值再复制给history,这是深层复制

第一部分“创建历史副本”的动机正确，但术语**需要纠正**：`params[:]` 只进行**浅拷贝（shallow copy）**，创建新的外层 list，但不递归复制嵌套的可变子对象。本课的 params 是由浮点数组成的一维 list，浮点数不可变，因此足以保存独立历史快照。

如优化器每次都**返回新列表**（本课 GD/Momentum/Adam 的 step 当前都是如此），直接 `history.append(params)` 也通常不会破坏先前记录；`params[:]` 额外支持返回相同列表且对其原地修改的 step 实现，较稳妥。

**独立实际运行测试通过：**
- 一维二次函数 $L(w)=(w-3)^2$，以 $w_0=5$、lr=0.1、3 个迭代步，完整轨迹为约 `[5.0, 4.6, 4.28, 4.024]`，共有 4 个历史记录；
- 对故意原地修改输入列表的测试优化器运行 3 步，历史快照依次为 `[3,2],[2,1],[1,0],[0,-1]`，不被后续更新污染；
- 对 `[[1,2],[3,4]]` 使用 `[:]` 后修改原始内层列表，浅拷贝中的嵌套列表也发生变化，验证没有递归复制。

**浮点校验细节：** 在教师单元测试中，严格使用 `== [4.024]` 曾因 Python 浮点结果为 `4.023999999999999` 而导致断言失败；改用 `math.isclose` 后所有断言通过。该失败是测试断言过严，并非学生算法错误。

尚未运行官方完整 5000 步的三优化器实验；这是下一环节，由学生先完成实验调用与结果解读。

In [ ]:
# optimize() 的独立回归检查（2026-11-06 教师实际运行成功）
import math

class _GDForOptTest:
    def __init__(self,lr): self.lr=lr
    def step(self,params,grads): return [p-self.lr*g for p,g in zip(params,grads)]

def _quad_grad(params): return [2*(params[0]-3)]

trajectory=optimize(_GDForOptTest(0.1), lambda p:(p[0]-3)**2, _quad_grad, [5.0],steps=3)
assert len(trajectory)==4
assert all(math.isclose(p[0], expected,rel_tol=1e-12) for p,expected in zip(trajectory,[5.,4.6,4.28,4.024]))
print("Quadratic trajectory:",trajectory)

class _InPlaceTest:
    def step(self,params,grads):
        for i in range(len(params)): params[i]-=grads[i]
        return params

snapshots=optimize(_InPlaceTest(),lambda p:None,lambda p:[1.]*len(p),[3.,2.],steps=3)
assert snapshots==[[3.,2.],[2.,1.],[1.,0.],[0.,-1.]]
print("In-place snapshots:",snapshots)

nested=[[1,2],[3,4]]
shallow=nested[:]
nested[0][0]=99
assert shallow[0][0]==99
print("PASS: trajectory, in-place history independence, and shallow-copy distinction")

### 下一项：三种优化器的统一实验调用（等待学生实现）

现在使用已完成的 `rosenbrock`、`rosenbrock_gradient`、`GradientDescent`、`SGDMomentum`、`Adam` 和 `optimize`。使用相同起点 `[-1.0,1.0]` 和 5000 个更新步、按官方原始示例分别设 GD lr=0.0005，SGDMomentum lr=0.0001 / momentum=0.9，Adam lr=0.01。

先让学生**独立填写三个调用**以及输出最终 `loss` 的循环，不提前展示最终数值。询问 `history` 为何长度 5001，以及最终损失要怎样从轨迹得到。待学生提交后实际运行并解释收敛，进入官方 Exercises 与 Use It / Ship It。

In [ ]:
start = [-1.0, 1.0]
steps = 5000

gd_history = optimize(
    GradientDescent(lr=0.0005),
    rosenbrock,
    rosenbrock_gradient,
    start,
    steps
)

sgd_history = optimize(
    SGDMomentum(lr=0.0001, momentum=0.9),
    rosenbrock,
    rosenbrock_gradient,
    start,
    steps
)

adam_history = optimize(
    Adam(lr=0.01),
    rosenbrock,
    rosenbrock_gradient,
    start,
    steps
)

## Build It · Step 5：官方三优化器比较（学生原始提交与检查）

**学生原始三个优化器调用：通过。** GD(lr=0.0005)、SGDMomentum(lr=0.0001, momentum=0.9)、Adam(lr=0.01)，`start=[-1.0,1.0]`、`steps=5000` 与官方原始示例完全匹配。

**学生理解题原始回答：**
> 1.因为历史保存了出事的param 2.rosenbrock[adam_history[-1][0]][adam_history[-1][1]] 就是最后的损失

1. 第 1 问正确：首次 `history = [params[:]]` 已经记录第 0 步，后续 5000 次更新各追加一次，所以长度为 5001。学生所说“出事”按语义应为“初始”。
2. 第 2 问知道需要把最后参数送给损失函数，思路正确但 Python 语法有误：`rosenbrock` 是接受 `params=[x,y]` 的函数，应使用 `rosenbrock(adam_history[-1])`，而非 `rosenbrock[...][...]` 的下标操作。

**教师独立实算**：用学生此前已验收的 GD、Momentum 更新公式，和只修正已确认括号错误的 Adam 公式，按官方配置完整运行 5000 次（Python 双精度）：

| 算法 | 最终参数 | 最终损失 |
|---|---|---|
| GD | [0.7981306984059844, 0.636103595905605] | 0.04083384590441455 |
| SGD+Momentum | [0.9404119287790969, 0.8841274606476993] | 0.0035568458096844665 |
| Adam | [0.9999994154698962, 0.9999988302237421] | 3.41726763964674e-13 |

三条轨迹均包含 5001 条记录。**这里三种算法学习率并不相同，属于官方示例配置比较，不是统一调参预算的严格公平最优比较。**

**绘图观察**：以训练步数作横轴、Rosenbrock loss 作纵轴，在对数纵轴下，Adam 的损失在第 2000 步附近达到约 `1.74e-19`，但第 2130 步反弹到约 `6.03e-3`，之后又回落，5000 步处为 `3.42e-13`。这说明最终 loss 不能代替过程稳定性分析。可能涉及常数学习率与 Adam 自适应历史统计量的动态耦合；具体机理需要进一步验证，不能凭曲线直接定因。

**前置条件**：历史 Notebook 中学生早期的 `SGDMomentum.__init__` 曾误写 `self.velocity=0`，后续教师测试版修正为 `None`；若学生本地仍保留旧构造器，运行本实验将报 `TypeError`。须确保在当前 Notebook 中先执行已修正的类定义，且各优化器在单次 5000 步实验中使用新的对象。

In [ ]:
# Matplotlib 绘图示例（教师教学代码；执行前先运行上面的历史轨迹单元）
import matplotlib.pyplot as plt

gd_losses = [rosenbrock(p) for p in gd_history]
sgd_losses = [rosenbrock(p) for p in sgd_history]
adam_losses = [rosenbrock(p) for p in adam_history]

plt.figure(figsize=(10, 6))
plt.plot(gd_losses, label="GD")
plt.plot(sgd_losses, label="SGD + Momentum")
plt.plot(adam_losses, label="Adam")
plt.xlabel("Optimization Step")
plt.ylabel("Rosenbrock Loss")
plt.title("Rosenbrock: GD vs Momentum vs Adam")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.show()

print("Final losses:", gd_losses[-1], sgd_losses[-1], adam_losses[-1])
print("Adam loss at steps 2000 and 2130:", adam_losses[2000], adam_losses[2130])

### 绘图知识点与下一轮理解检查（等待作答）

- `plt.plot(y)` 默认以 0、1、2、……作为 x，传入每一步计算出的 loss 作为 y。
- `plt.yscale("log")` 使用对数显示数量级，不会修改真实的 loss 数组；对数轴不能直接表示 0 或负数。
- `plt.legend()` 显示标签、`plt.grid(True)` 显示网格、`plt.xlabel()/ylabel()` 设置坐标轴文本。
- 训练过程观察指标不仅有最后一轮损失，还包括损失尖峰、波动、是否曾达到更优的中间状态。
- **等待学生回答**：在对数纵轴图中观察 Adam 约 2130 步的尖峰，为什么这意味着只记录最终 `loss` 不足以判断过程是否稳定？如在 Notebook 运行图出现异常，优先检查 `SGDMomentum.velocity` 初始化为 `None`。

官方课程继续保持 **01/08 进行中**；后续 Use It/Exercises/Ship It/Post Quiz 仍未完成，不向根 `LEARNING.md` 写入完成进度。

## Adam Loss Spike：学生理解检查与诊断（2026-11-06）

**学生原始回答：**

> 因为我们到达最优点附近的时候我们的损失后面会震荡,就是再最优点附近来回跳动,所以单纯的看第5000步时不行的,要观察整个loss曲线的变化

**验收：通过。** 学生理解：最终低 Loss 不能表示训练中全部时刻都稳定，必须看全过程。进一步的严谨补充：“最优点附近来回震荡”是关于参数轨迹的解释，而不仅是对 Loss 曲线的直接描述，必须检查参数位置、更新范数与梯度历史来验证。

**教师进一步独立数值诊断（实际 Python 运算；官方数学表达式顺序，均为 double precision）：** Rosenbrock 初始点 `[-1,1]`，Adam lr=0.01、betas=(0.9,0.999)、eps=1e-8，解析梯度 `[-2*(1-x)+200*(y-x**2)*(-2*x), 200*(y-x**2)]`，每步按公式保存新参数。

| 更新步 | L(theta_t) | ||theta_t - theta_(t-1)||_2 | theta_t |
|---|---:|---:|---|
| 2000 | 1.7412779644882e-19 | 2.016612556551002e-11 | [0.9999999995830152, 0.9999999991644446] |
| 2068 | 9.686688154886558e-21 | 5.116400095514484e-12 | [0.9999999999073764, 0.9999999998114248] |
| 2130 | 0.006030775800602256 | 0.00634134296016433 | [1.0028767633676872, 0.9980013256920274] |
| 2131 | 0.005923839152222576 | 0.006975143589623441 | [0.9971206329093646, 1.0019408168067065] |

这些数据不仅说明 loss 升高，还说明在该时间段的参数更新步长明显变大，且相邻步参数跨过最优点附近的两侧，支持局部摆动的描述。没有随机 mini-batch，仍发生 spike；因此不能简单归因于采样噪声。发生的具体动力学机制需要考察 Momentum / 二阶矩历史、谷底曲率及有限步长，暂不对某个因素作单因归因。

**复核口径提示**：接近最优点的混沌/数值敏感轨迹对梯度表达式的浮点运算顺序、具体实现有敏感性，因此跨实现的末步值可能略有差别；本段同一独立运行轨迹的 2000、2068、2130、2131 数字内部一致。

In [ ]:
# 教师诊断脚本：检查 Adam 的 loss 与每一步参数更新范数（需先已生成 adam_history）
import math
spike_steps = [2000, 2068, 2130, 2131]
for t in spike_steps:
    loss = rosenbrock(adam_history[t])
    step_norm = math.dist(adam_history[t], adam_history[t-1])
    print(f't={t}, loss={loss:.6e}, step_norm={step_norm:.6e}, params={adam_history[t]}')
# 注意：该检查针对已有的 adam_history 进行，不重复调用 Adam.step() 破坏优化器内部状态。

### 下一道互动题（补充诊断练习，不计官方 Exercises）

要求学生使用已有的 `adam_history`，自己写出 `t=2068` 与 `t=2130` 两步的：
1. `rosenbrock(adam_history[t])` 计算当前 Loss；
2. `math.dist(adam_history[t], adam_history[t-1])` 计算单步参数变化的欧氏范数。

然后解释：如果单步更新范数突然从约 `5e-12` 增大到约 `6e-3`，这比仅看 loss 更能说明什么？之后转入官方 Learning Rate Sweep 练习，按一题一问推进。

本课仍在学习中，暂不更新 `LEARNING.md` 整课完成记录。

In [ ]:
import math

for t in [2068, 2130]:
    loss = adam_losses[t]
    step_norm = math.dist(sgd_history[t],sgd_history[t-1])

    print(t, loss, step_norm)

## Adam 参数更新范数：学生代码验收与概念订正（2026-11-06）

**学生原始解释（保持原文）：**

> 能说明我的-lr*梯度变大了 我们应该减少lr

**原始代码：部分正确，需修正。** 学生正确使用 `adam_losses[t]` 获取 Adam Loss、`math.dist` 计算欧氏距离，但 `math.dist(sgd_history[t],sgd_history[t-1])` 用了 **SGD+Momentum** 轨迹。导致图表对比的是 Adam 的 loss 和 SGD 的参数步长，不能分析 Adam 的振荡。应改用 `math.dist(adam_history[t], adam_history[t-1])`。

**教师独立实跑**：从 Rosenbrock 的初始点 `(-1,1)` 出发，Adam lr=0.01、beta=(.9,.999)、epsilon=1e-8；SGD+Momentum lr=.0001、momentum=.9；均为 5000 步，以 Python double precision 算术重算，关键数据：

| step | Adam loss | Adam ||Δθ||₂ | SGD+Momentum ||Δθ||₂ |
|---:|---:|---:|---:|
| 2068 | 9.686688154886558e-21 | 5.116400095514484e-12 | 0.0002925995712327372 |
| 2130 | 0.006030775800602256 | 0.00634134296016433 | 0.0002788943603366522 |

**概念订正：** 普通 GD 才是 `Δθ=-lr*g`；Adam 是

$$
\Delta\theta_t=-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$

这里 `lr` 在所有迭代中保持常数 0.01；更新范数显著增加，说明相应时刻自适应更新的范数增加，不能简单解释为“学习率变大”或“当前梯度直接乘学习率变大”。当前梯度会通过 EMA 影响一阶、二阶矩，但三者关系需区分。学生提出**减少学习率**可作为一项可检验假设；尚不能确定是唯一原因，也不保证消除尖峰。

本题属于额外诊断练习。验收状态：`loss` 提取通过，`step_norm` 应修正轨迹，降低 lr 的建议可用实验验证；记录保持完整。

**课程正式进度：** Build It Step 5 已完成主要实现与官方三方法运行。下一步进入官方 Exercises 第 1 题 Learning Rate Sweep（GD 的五个学习率）。

In [ ]:
# 教师可复现核验：这段代码要求之前已创建 adam_history、sgd_history、adam_losses
import math
for t in [2068, 2130]:
    adam_loss = adam_losses[t]
    adam_norm = math.dist(adam_history[t], adam_history[t-1])
    sgd_norm = math.dist(sgd_history[t], sgd_history[t-1])
    print(f"step={t}: Adam loss={adam_loss:.6e}, Adam step norm={adam_norm:.6e}, SGD step norm={sgd_norm:.6e}")
# 检验历史轨迹长度
assert len(adam_history) == len(sgd_history) == 5001

### 下一项官方 Exercise 1：Learning Rate Sweep（等待学生独立实现）

仅使用 `GradientDescent`（**不是 Adam**），学习率分别取 `0.0001,0.0005,0.001,0.005,0.01`，初始点 `[-1.0,1.0]`，各运行 5000 步，计算最后一次历史参数的 `rosenbrock` 损失。对于过大学习率，直接执行未做保护的官方 `optimize` 可能溢出，须在实验中明确报告异常，不把异常冒充有意义的 loss。

建议学生补全一段短循环，每个学习率实例化**新的** `GradientDescent` 并调用 `optimize`；若遇数值异常，让学生提出合理处理策略。按一问一答，先验收实现，再解释不同 lr 的曲线，并与固定学习率/衰减策略作对比。

In [ ]:
learning_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]

for lr in learning_rates:
    optimizer = GradientDescent(lr=lr)

    history = optimize(
            optimizer,
            rosenbrock,
            rosenbrock_gradient,
            start,
            steps
        )

    final_loss = rosenbrock(history[-1])

    print(f"lr={lr}, loss={final_loss}")

## 官方 Exercise 1：Learning Rate Sweep — 学生提交与溢出诊断（2026-11-06）

**学生提交：**见前一代码单元，依官方要求枚举 5 个学习率，逐次新建 `GradientDescent`，使用已有 `optimize`、`rosenbrock`、`rosenbrock_gradient`、`start` 和 `steps=5000`。代码逻辑正确，但在其中一个学习率上触发数值溢出，循环中断。这是本练习的重要实验现象，不是编码错误。

**学生实际错误原文关键部分：**
```text
OverflowError: (34, 'Numerical result out of range')
  history = optimize(...)
  grads = grad_func(params)
  df_dx = -2 * (1 - x) - 400 * x * (y - x ** 2)
```

**教师在独立 Python 环境复现**（普通 Python float、相同解析梯度表达式、起点 `[-1.0, 1.0]`、5000 步、三行官方 GD 定义）：

| lr | 状态 | 最终 loss |
|---:|---|---:|
| 0.0001 | 完成 | 2.09718512762 |
| 0.0005 | 完成 | 0.0408338459044 |
| 0.001 | 完成 | 0.00362713754709 |
| 0.005 | 第 11 次更新前在梯度计算 `x ** 2` 时发生 `OverflowError` | 不存在有效最终 loss |
| 0.01 | 迭代中参数变为 `inf`，后续传播为 `nan`，普通循环运行到结束 | NaN（无效） |

对应复现实验中 `lr=0.005` 更新 10 次后的坐标为约 `[-7.426800277e156, 2.397992568e104]`；第 11 次梯度计算溢出。`lr=0.01` 第 9 次更新首次产生非有限参数 `[inf, 9.497665727e226]`，但 Python float 流水线没有保证立即抛异常，所以必须显式使用 `math.isfinite` 防止将 NaN 误认为正常完成。

**概念解释：** Rosenbrock 梯度 `-2*(1-x)-400*x*(y-x**2)` 含有三次增长量级的项。大步长使参数离开稳定区域后，梯度与更新量互相放大，出现数值发散。最终 `OverflowError` 是计算机浮点表示极限所致；数学上的迭代可能早已发散，不能将异常作为唯一判据。

**本轮评价：** 学习率枚举、调用与 Loss 读取正确；尚需学生加入单次 `lr` 的异常处理，并判断 `math.isfinite(final_loss)`。异常和 NaN 均应分别报告为失败，不生成虚构最终 loss，也不要简单吞掉错误。此后比较稳态轨迹、绘制 loss 曲线，完成官方 Exercise 1。

In [ ]:
# 教师独立重现实验及可复现日志（不修改学生原始代码）
import math
for lr in [0.0001, 0.0005, 0.001, 0.005, 0.01]:
    gd = GradientDescent(lr=lr)
    try:
        hist = optimize(gd, rosenbrock, rosenbrock_gradient, [-1.0, 1.0], 5000)
        last = rosenbrock(hist[-1])
        if math.isfinite(last):
            print(f"lr={lr}: OK, final_loss={last:.12g}")
        else:
            print(f"lr={lr}: NONFINITE final loss = {last!r} (diverged)")
    except (OverflowError, ValueError) as err:
        print(f"lr={lr}: ERROR {type(err).__name__}: {err} (diverged)")

### 下一次互动任务：学生自己添加失败状态处理

只修改学生已有的 `for lr` 循环，在每个学习率外加 `try/except (OverflowError, ValueError)`，正常返回后使用 `math.isfinite(final_loss)` 将正常收敛与非有限结果区分。暂不改变公式或把损失上升当成异常。完成后再进入绘制稳定三个学习率的曲线与失稳解释。本课尚未完成，不更新根目录 `LEARNING.md`。

In [ ]:
learning_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]

import math

for lr in learning_rates:
    optimizer = GradientDescent(lr=lr)

    try:
        history = gd_history = optimize(
            GradientDescent(lr=lr),
            rosenbrock,
            rosenbrock_gradient,
            start,
            steps
        )
        final_loss = rosenbrock(history[-1])

        if math.isfinite(final_loss):
            print(f"lr={lr}, loss={final_loss}")
        else:
            print(f"lr={lr}, diverged: non-finite loss")

    except (OverflowError, ValueError) as e:
        print(f"lr={lr}, diverged: {e}")

## 官方 Exercise 1：学习率扫描异常处理——学生提交与验收（2026-11-06）

学生原始解释：
> 因为可能计算出来的值不是有限值,但是没有抛出异常

**理解检查：正确。** Python 浮点计算可产生 `nan` 或 `inf` 而不抛出异常，故 `try/except` 用于显式异常，`math.isfinite` 用于有限值判定，两者互补。

**代码通过本轮异常处理要求。** 五个学习率逐个尝试，`except (OverflowError,ValueError)` 使扫描能继续运行，正常返回则读取最后一组参数并检查结果有限性。

**两处代码可改善：**
1. `optimizer = GradientDescent(lr=lr)` 赋值后没有使用；调用 `optimize()` 时又创建了新的 `GradientDescent(lr=lr)`。可直接传入 `optimizer`。
2. `history = gd_history = optimize(...)` 同时覆盖之前三优化器比较得到的 `gd_history`；除非明确需要更新旧变量，此处用 `history = optimize(...)` 更合适。后续保存多组 loss 轨迹应使用字典。

**独立复现实跑（教师执行原始代码，起点 [-1,1]、5000 步，Rosenbrock）：**
```text
lr=0.0001, loss=2.0971851276168345
lr=0.0005, loss=0.04083384590441455
lr=0.001, loss=0.0036271375470905893
lr=0.005, diverged: (34, 'Numerical result out of range')
lr=0.01, diverged: non-finite loss
```
确认这一轮的学习率扫描运行逻辑无误，但官方 Exercise 1 的“Plot or print the final loss”已经满足（已逐项打印最终 loss 和失败状态），扩展绘制 loss 随步数变化的曲线可以作为加深理解的补充。**结论只限本 Rosenbrock 配置，不代表一般最优学习率。**

待学习内容：官方其余三道 Exercises、Use It、Ship It、Post Quiz。

In [ ]:
# 教师实跑验证日志，复现学生的原始异常处理逻辑
learning_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]
for lr in learning_rates:
    optimizer = GradientDescent(lr=lr)
    try:
        history = gd_history = optimize(
            GradientDescent(lr=lr), rosenbrock, rosenbrock_gradient, start, steps
        )
        final_loss = rosenbrock(history[-1])
        if math.isfinite(final_loss):
            print(f"lr={lr}, loss={final_loss}")
        else:
            print(f"lr={lr}, diverged: non-finite loss")
    except (OverflowError, ValueError) as e:
        print(f"lr={lr}, diverged: {e}")
# 注意：会覆盖已有 gd_history，因此仅作为教师复现记录，不应当与实验主 Notebook 变量混用。

### 下一步：保存多组学习率 Loss 曲线（补充 Matplotlib 练习）

引导学生在扫描开始前创建 `loss_curves = {}`，只在 `math.isfinite(final_loss)` 时，使用 `[rosenbrock(p) for p in history]` 把本轮历史转换为 Loss 序列，存入 `loss_curves[lr]`；各学习率必须实例化自己的 `GradientDescent`。然后逐条绘制曲线并使用 `plt.yscale("log")` 方便比较多个数量级。过程仍然一问一答，学生先独立实现字典保存部分，不直接填好整段绘图答案。

此阶段 Exercise 1 的官方打印最终 loss 要求已满足，可继续练习其他 Exercise，但按用户主动请求“教我画曲线”优先做小段绘图练习。

In [ ]:
learning_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]

import math

loss_curves = {}

for lr in learning_rates:
    optimizer = GradientDescent(lr=lr)

    try:
        history = optimize(
            optimizer,
            rosenbrock,
            rosenbrock_gradient,
            start,
            steps
        )
        loss_curves[lr] = [rosenbrock(h) for h in history]
        final_loss = rosenbrock(history[-1])

        if math.isfinite(final_loss):
            loss_curves[lr] = [rosenbrock(h) for h in history]
            print(f"lr={lr}, loss={final_loss}")
        else:
            print(f"lr={lr}, diverged: non-finite loss")

    except (OverflowError, ValueError) as e:
        print(f"lr={lr}, diverged: {e}")

## Learning Rate Sweep — 保存每条 Loss 曲线（2026-11-09）

**学生代码：列表推导式与字典结构正确，验收仍需一处逻辑订正。** 其余部分保留此前正确的每 lr 新建优化器、`try/except` 与 `math.isfinite` 检查。

学生直接使用 `loss_curves[lr] = [rosenbrock(h) for h in history]` 正确从参数轨迹计算损失序列，但在 `final_loss = ...` 和 `math.isfinite` 检查**之前**重复保存了一次，后面条件分支又保存了一次。应删除第一行，只在 `math.isfinite(final_loss)` 成立时保存，最好再确认曲线各点为有限值。否则如果后续某次运行没有抛异常、但出现 `nan`，无效 loss 曲线会先进入字典且不会被移除。

示例订正：
```python
history = optimize(optimizer, rosenbrock, rosenbrock_gradient, start, steps)
final_loss = rosenbrock(history[-1])
if math.isfinite(final_loss):
    loss_curves[lr] = [rosenbrock(p) for p in history]
    print(f"lr={lr}, loss={final_loss}")
else:
    print(f"lr={lr}, diverged: non-finite loss")
```

**教师实际复现的补充细节**：使用 Notebook 学生此前提交的梯度表达式 `-2*(1-x)-400*x*(y-x**2)` 和普通 Python float，两个高学习率 `0.005`、`0.01` 均在梯度计算时抛 `OverflowError`，字典最终仅包含 `0.0001,0.0005,0.001`。此前记录的 `lr=0.01` 以 `NaN` 结束，来自另一种代数等价但浮点运算顺序不同的 `-2*(1-x)+200*(y-x**2)*(-2*x)` 实现。失稳时的 `inf/nan/OverflowError` 表现会随运算顺序不同而变化；上述两种都属于发散，不应把某种错误形式当成固定性质。两个路径都不能视为有效收敛。

这次教师复现仅验证当前原始扫描代码，当前理解题尚需学生按要求删除重复赋值，再自行完成 `plt.plot` 多曲线绘图。整课 01/08 仍在进行中。

In [ ]:
# 教师验证示例：在当前 Notebook 单独运行下面代码不会影响历史的优化器实例
# 假设已定义 GradientDescent / optimize / rosenbrock / rosenbrock_gradient / start / steps
# 这份示例只保留有效曲线，供后续绘图练习使用
import math
stable_curves = {}
for lr in [0.0001, 0.0005, 0.001, 0.005, 0.01]:
    try:
        result = optimize(GradientDescent(lr=lr), rosenbrock, rosenbrock_gradient, [-1.,1.], 5000)
        loss_series = [rosenbrock(p) for p in result]
        if all(math.isfinite(v) for v in loss_series):
            stable_curves[lr] = loss_series
            print(f"{lr}: valid ({len(loss_series)} points), last={loss_series[-1]:.9g}")
        else:
            print(f"{lr}: nonfinite loss series (exclude)")
    except (OverflowError, ValueError) as err:
        print(f"{lr}: {type(err).__name__} (exclude)")
assert list(stable_curves) == [0.0001, 0.0005, 0.001]

### 下一步：Matplotlib 多曲线绘图（等待学生）

让学生先删掉重复的 `loss_curves[lr] = ...`，重新执行存储单元，确认 `loss_curves.keys()` 只有稳定的三个学习率。再分段练习：

```python
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for lr, losses in loss_curves.items():
    # 学生独立填写：plt.plot(???, label=???)
    pass
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Rosenbrock loss")
plt.legend()
plt.grid(True)
plt.show()
```

理解提问：`loss_curves.items()` 为什么比直接遍历字典 keys 更适合这里？要等学生回答并画图之后，再进入官方 Exercise 2：Momentum Comparison。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

for lr, losses in loss_curves.items():
    plt.plot(losses, label=f"lr={lr}")

plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Rosenbrock Loss")
plt.title("GD Learning Rate Comparison")
plt.legend()
plt.grid(True)
plt.show()

## 官方 Exercise 1 补充：Matplotlib 学习率曲线（2026-11-09）

学生独立提交的绘图代码见上方代码单元。**代码正确，通过验收。** `loss_curves.items()` 同时提供学习率与完整损失列表；`plt.plot(losses)` 默认以序列下标表示迭代步；f-string `label=f"lr={lr}"` 构造各条曲线图例；`plt.yscale("log")` 用对数纵轴展示多个数量级。

**导师独立验证**：使用 Python Matplotlib 的非交互 Agg 后端，输入三组示例正数 Loss（不同于学生本地真实 Rosenbrock 轨迹），三条 Line2D 对象及标签 `lr=0.0001`、`lr=0.0005`、`lr=0.001` 全部正确，纵轴 scale 为 `log`，检查通过。此验证不能替代用户本地真实运行的曲线图及轨迹证据；未声称已看见用户的图像。

**重要前置状态**：上一轮学生创建 `loss_curves` 的代码曾在有限值检查前重复保存，应先按讲评删除条件之前的赋值，并重新执行该单元；此轮绘图代码本身没有错误，逻辑上应接已修正的字典数据。

**下一项为官方 Exercise 2 Momentum Comparison**，参数分别为 `[0.0, 0.5, 0.9, 0.99]`，记录完整 Loss 轨迹，比较收敛速度与过冲/振荡行为。与官方 Exercise 1 一样，用相同初始点、相同步数，逐个新建优化器。官方尚未完成的练习、Use It、Ship It 与 Post Quiz 仍需逐项完成。

### 下一轮独立代码任务：Momentum 比较的字典与循环（未提交）

沿用前面的 `optimize`、`rosenbrock`、`rosenbrock_gradient`、`start=[-1,1]`、`steps=5000`。固定官方示例的 `lr=0.0001`，分别用 `momentum in [0.0,0.5,0.9,0.99]` 构造新 `SGDMomentum`，计算并保存完整 Loss 曲线至 `momentum_curves[momentum]`。学生先实现循环，不提前告诉最终数值或“哪一个最好”。在本节理解检查中说明为什么不能把同一个 SGDMomentum 对象用于四种 momentum 配置：历史速度状态不应共享。

In [ ]:
momentum_values = [0.0, 0.5, 0.9, 0.99]
momentum_curves = {}

for momentum in momentum_values:
    optimizer = SGDMomentum(
        lr=0.0001,
        momentum=momentum
    )

    history = optimize(
        optimizer,
        rosenbrock,
        rosenbrock_gradient,
        start,
        steps
    )

    momentum_curves[momentum] = [rosenbrock(h) for h in history]

    print(
        f"momentum={momentum}, "
        f"final_loss={momentum_curves[momentum][-1]}"
    )

## 官方 Exercise 2：Momentum Comparison 实验代码验收（2026-11-09）

**学生原始解释：**
> 因为我们的旧的对象都包含了旧的v,我们要给他置零

**判定：代码结构正确，理解检查正确。** 四种 `momentum=[0,0.5,0.9,0.99]` 每次创建新实例，避免带入旧 `velocity`；同起点 `[-1,1]`、同 `lr=0.0001`、同 5000 步，`momentum_curves[momentum]` 保存完整损失轨迹。重要前提：本地 `SGDMomentum` 构造器须确保 `self.velocity=None`；若还沿用最早的错误版 `=0`，则本地运行会报 TypeError。

**教师独立使用此前同样的数学实现复现**：Python 双精度，Rosenbrock 梯度 `[-2*(1-x)-400*x*(y-x**2),200*(y-x**2)]`，动量 `v=beta*v+g`、`p=p-lr*v`。

| Momentum | 5000步最终 Loss | 最小 Loss | 相邻两步 Loss 上升次数 |
|---:|---:|---:|---:|
| 0 | 2.0971851276168345 | 2.0971851276168345 | 0 |
| 0.5 | 0.4061988349429453 | 0.4061988349429453 | 0 |
| 0.9 | 0.0035568458096844665 | 0.0035568458096844665 | 0 |
| 0.99 | 3.0419989306345348e-21 | 1.7612214676686442e-22 | 930 |

**独立实际采样结果（Loss，不是插值）：**

| step | beta=0 | beta=.5 | beta=.9 | beta=.99 |
|---:|---:|---:|---:|---:|
| 500 | 3.831294315425863 | 3.6653890152787834 | 2.1465227106672726 | 0.046674078824096944 |
| 1000 | 3.6646935837261263 | 3.317344641305837 | 0.4106539859496047 | 0.0006557383632197774 |
| 2000 | 3.316577384567678 | 2.541568203815134 | 0.07353412455308 | 2.4288148668530935e-09 |
| 3000 | 2.9442006335149724 | 1.6129573649207376 | 0.0235097983859702 | 3.9685902841355127e-13 |
| 4000 | 2.540581463304661 | 0.7614819606904855 | 0.008815019076222551 | 6.929960964236994e-17 |
| 5000 | 2.0971851276168345 | 0.4061988349429453 | 0.0035568458096844665 | 3.0419989306345348e-21 |

其中 beta=.99 最终损失最低，但共 930 个相邻步的 Loss 上升；这是一种局部波动指标，不足以单独确定发生了多少次物理意义上的“过冲”。由于使用非归一化动量 `v_t=beta*v_{t-1}+g_t`，不同 beta 对梯度历史的累计权重不同，特别在稳定常值梯度近似下 `v\approx g/(1-beta)`；这让有效步长随 beta 改变。当前实验控制了名义 lr，而非相同的有效步长。

**需要完成的官方要求：** 学生已保存每一步 Loss，但尚未亲自绘图与解释哪个 beta 最快、哪里上升或波动。下一轮继续绘图和结果解读，再正式完成 Exercise 2。

In [ ]:
# 教师复现/验证：基于学生已创建的 momentum_curves，输出官方指标
# 如果本地已运行四个实验，不需要重复训练
for beta, losses in momentum_curves.items():
    rises = sum(losses[i + 1] > losses[i] for i in range(len(losses) - 1))
    assert len(losses) == 5001
    print(f'beta={beta}: final={losses[-1]:.12g}, min={min(losses):.12g}, rising_steps={rises}')

# 可选：检查几个固定时刻，需本地依次执行之前的优化器和 optimize 定义
for t in [500, 1000, 2000, 3000, 4000, 5000]:
    print(t, {beta: momentum_curves[beta][t] for beta in [0.0, 0.5, 0.9, 0.99]})

### 下一轮一问一答：绘制 Momentum 曲线并解释

沿用上一练习的 Matplotlib 方法，用户自行在 `momentum_curves.items()` 上迭代，`plt.plot(losses, label=f"momentum={momentum}")`，使用对数纵轴观察四种 beta 的轨迹。请同时说出：
1. 5000 步预算下哪个 beta 的最终 Loss 最低？
2. beta=.99 为什么在收敛很快的同时仍出现 Loss 上升？可以先作关于历史速度惯性的假设，再用完整曲线解释，避免直接断言唯一原因。
3. 只固定 lr 而改变 beta，是否意味着每种优化器的有效更新规模相同？（提示：无归一化动量稳态权重之和约 1/(1-beta)）

本课保留进行中：Exercise 2 的轨迹生成已通过，最终解读待学生作答，其余 Exercise 3/4、Use It、Ship It 和 Post Quiz 尚未完成。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

for momentum, losses in momentum_curves.items():
    # 你来画曲线
    plt.plot(losses, label = f"momentum={momentum}")

plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Rosenbrock Loss")
plt.legend()
plt.grid(True)
plt.show()

## 官方 Exercise 2：Momentum 曲线绘制与概念解释（2026-11-09）

**学生原始解释：**
> 因为我们的整体的g是梯度近似恒定的,所以可以快速下降,但是因为动量太大了,有的时候不能一次就把动量掉头,有可能上升

**验收结论：绘图代码通过；概念主要正确，稳态假设需修正。** Matplotlib 代码遍历 `momentum_curves.items()`，使用 `plt.plot(losses,label=f"momentum={momentum}")` 绘制各条曲线；对数纵轴、坐标和图例设置正确。学习者并未上传本地实际绘图截图，因此此处仅核验代码。教师已独立运行 Python 数值迭代并复核四种 Momentum 的真实结果。

对于动量 $v_t=\beta v_{t-1}+g_t$，较大 $\beta$ 使历史方向保留时间更长，在持续相同梯度方向上能累积更大的速度，但梯度反向时未必能迅速改变速度方向。例如 $v_{t-1}=10$、$g_t=-3$，$\beta=0.99$ 得到 $v_t=6.9>0$，仍按原来的方向更新。相对地 $\beta=0.5$ 时 $v_t=2>0$，历史影响衰减较快。这能解释过冲/局部 loss 上升的可能性。

必须区分推导假设：$v_t\approx g/(1-\beta)$ 需要常数（或近似恒定）梯度，不表示 Rosenbrock 的实际梯度“整体近似恒定”；其梯度依赖位置，尤其在弯曲谷地会改变方向。较大 $\beta$ 的实际速度及损失变化需由轨迹观察。另需注意恒定 lr 下未归一化动量的有效更新尺度随 $\beta$ 改变，因此不能仅将性能差异归因于惯性。

**教师复核数值（Python float，同一 `start=[-1,1]`、5000 步、lr=.0001）：**
| beta | 最终 Loss | 上升步数（相邻 $L_{t+1}>L_t$） |
|---:|---:|---:|
| 0.0 | 2.0971851276168345 | 0 |
| 0.5 | 0.4061988349429453 | 0 |
| 0.9 | 0.0035568458096844665 | 0 |
| 0.99 | 3.0419989306345348e-21 | 930 |

与官方 Exercise 2 的目标一致：固定名义学习率比较四个动量，观察收敛差异和局部损失上升。**此练习完成**；后续对优化器的泛化结论仍需更广泛实验验证。

**课程整体状态：** Exercise 1、2 已完成。Exercise 3、4、Use It、Ship It、Post Quiz 尚待完成；不能更新根目录 `LEARNING.md` 的整课完成记录。

### 官方 Exercise 3：Saddle Point Escape（概念引入，等待学生作答）

官方题目：定义 $f(x,y)=x^2-y^2$（原点是鞍点），从 $(0.01,0.01)$ 开始比较 vanilla GD、SGD with momentum、Adam 的行为。问：哪些能够离开鞍点附近？

先做一步概念推导，再实现函数、梯度、运行实验。沿两个坐标轴分别观察：

$$f(x,0)=x^2\ge0,\qquad f(0,y)=-y^2\le0.$$

原点的梯度为零，但不是局部最小值；不同方向呈现相反曲率。

**本轮一问一答（等待学生）：** 独立写出 $\partial f/\partial x$、$\partial f/\partial y$，求初始点 $(0.01,0.01)$ 的梯度，并用一句话说明原点为什么是鞍点。不要提前给出梯度答案。下一轮验收后由学生独立实现 Python 代码，再设相同步数和清晰逃逸阈值比较。

## 官方 Exercise 3：Saddle Point Escape — 梯度理解检查（2026-11-09）

**学生原始回答：**
> 1.f对于x的偏导= 2x f对于y的偏导 = -2y 2.梯度向量(0.02,-0.02) 3.不能,因为在(0, 0)这个点的梯度为(0,0)

**验收：三题 3/3 正确。** 给定 $f(x,y)=x^2-y^2$，梯度为 $\nabla f(x,y)=(2x,-2y)$，在 $(0.01,0.01)$ 处为 $(0.02,-0.02)$。若精确从原点出发，且使用确定性精确梯度、GD 无额外扰动，则梯度为零、更新量为零，迭代会停留在原点。

原点处 Hessian $\operatorname{diag}(2,-2)$ 有一正一负两个特征值，因而是**严格鞍点**。若从初始点 $(0.01,0.01)$ 出发，GD 学习率 $\eta=0.01$ 时一轮后 $x_1=0.0098$、$y_1=0.0102$；沿 x 方向靠近零，沿 y 方向离开零。题目“逃逸”指远离鞍点附近，函数在 y 方向无下界，不能将逃逸误认为收敛至有限全局最优。

**当前状态：** 理解检查通过，官方 Exercise 3 尚需学生独立实现函数与梯度，并运行 GD、Momentum、Adam 作实际对比，不能提前标记整题完成。

### 下一项独立编程任务：Saddle 函数与梯度

请学生只实现：
```python
def saddle(params):
    x, y = params
    # return f(x,y)

def saddle_gradient(params):
    x, y = params
    # return [df_dx, df_dy]
```

学生提交后先做解析点检查（例如 $(0,0)$ 与 $(0.01,0.01)$），再使用已完成的 `optimize` 运行 GD、Momentum、Adam。保持一问一答，暂不提前代写完整三优化器运行脚本或答案。根目录 `LEARNING.md` 不标记完成。

In [ ]:
def saddle(params):
    x, y = params
    # 返回 f(x,y)
    return x ** 2 - y ** 2

def saddle_gradient(params):
    x, y = params
    # 返回包含两个偏导数的列表
    return [2 * x, -2 * y]

## 官方 Exercise 3：学生实现与 Hessian 概念追问（2026-11-09）

学生针对此前讲解主动追问：
> 你这个结论是怎么得出来的

指的是：为何 $H=\begin{pmatrix}2&0\\0&-2\end{pmatrix}$ 有特征值 $2,-2$，可以判断原点是严格鞍点。

**学生提交的函数实现：两函数均正确。** `saddle([x,y])=x**2-y**2`；`saddle_gradient([x,y])=[2*x,-2*y]`。教师实际运行确认 `saddle([0,0])==0`、`saddle_gradient([.01,.01])==[.02,-.02]`（浮点比较容差内）。

**数学讲解记录：**
1. 一阶偏导 $\partial_x f=2x,\partial_y f=-2y$，二阶偏导 $\partial^2_{xx} f=2,\partial^2_{yy}f=-2$；混合偏导皆为零。得到 Hessian $H_f=\operatorname{diag}(2,-2)$。
2. 特征方程 $\det(H-\lambda I)=(2-\lambda)(-2-\lambda)=0$，解得 $\lambda_1=2,\lambda_2=-2$。相应单位特征方向是 x 轴、y 轴。
3. 沿单位方向 $d$ 的二阶方向导数等于 $d^THd$；沿 Hessian 单位特征向量即特征值。x 轴方向正曲率，y 轴方向负曲率。
4. 原点梯度零，但任何 $\delta\ne0$ 都有 $f(\delta,0)=\delta^2>0=f(0,0)$ 和 $f(0,\delta)=-\delta^2<0$，任意小邻域内都有更大和更小函数值，不是局部最大或局部最小；原点是鞍点。
5. 优化理论的严格鞍点条件：驻点处 Hessian 存在严格负特征值；本题同时存在正、负特征值且 $\lambda_{\min}=-2<0$，所以是严格鞍点。一般 Hessian 存在零特征值时二阶测试可能不充分，不能一概判断。

**实际数值核对：** Python `numpy.linalg.eigh([[2,0],[0,-2]])` 得到 eigenvalues `[-2,2]`；测试方向 $(1,0)$ 与 $(0,1)$ 时 $d^THd$ 分别为 $2,-2$；`f(.01,0)=0.0001`，`f(0,.01)=-0.0001`。

**状态：** Saddle 函数与解析梯度实现已验收，但官方 Exercise 3 尚需对三种优化器的逃逸行为实际对比，尚未完成。

In [ ]:
# Hessian 定义与鞍点方向性检查：教师已在独立 Python 环境实际运行
import math
import numpy as np

assert saddle([0.0, 0.0]) == 0.0
assert all(math.isclose(a, b, abs_tol=1e-12) for a,b in zip(
    saddle_gradient([0.01, 0.01]), [0.02, -0.02]
))
H = np.array([[2.0, 0.0], [0.0, -2.0]])
eigvals = np.linalg.eigvalsh(H)
assert np.allclose(eigvals, [-2.0, 2.0])
e_x, e_y = np.array([1., 0.]), np.array([0., 1.])
assert math.isclose(float(e_x @ H @ e_x), 2.0)
assert math.isclose(float(e_y @ H @ e_y), -2.0)
assert saddle([0.01, 0.0]) > 0 > saddle([0.0, 0.01])
print("PASS: function, gradient, Hessian eigenvalues, and saddle directions")

### 下一步互动任务（等待学生）

先完成一个最小理解检查：若把函数改为 $h(x,y)=x^2+y^2$，请学生自行求 Hessian 和两个特征值，并说明原点是什么类型的驻点。验收后调用已实现的 `optimize`，用 `start=[0.01,0.01]`、相同步数和事先定义的逃逸指标对 GD、Momentum、Adam 作对比。

课堂应避免一次性代写完整实验；记录学生原始答案和实际运行输出。

## Hessian 与向量记号阅读：学生原始回答（2026-11-09）

> \[
> H=
> \begin{pmatrix}
> 2&0\\
> 0&2
> \end{pmatrix}
> \]
> 特征值是 2, 2 是局部最小点 我现在知道我是哪里不理解了,我是对这些公式符号不理解
> \[
> f(\boldsymbol\delta)
> =f(0)+\nabla f(0)^\top\boldsymbol\delta
> +\frac12\boldsymbol\delta^\top H\boldsymbol\delta
> \]
> \[
> \frac{d^2}{dt^2}f(\boldsymbol x+t\boldsymbol d)
> \bigg|_{t=0}
> =\boldsymbol d^\top H_f(\boldsymbol x)\boldsymbol d
> \]

**验收：正确。** 对于 \(h(x,y)=x^2+y^2\)，Hessian 为 \(\mathrm{diag}(2,2)\)、特征值为 \(2,2\)，原点既是严格局部最小值也是严格全局最小值。学生明确反馈：困难在符号阅读，不在于求导。

## 公式解码：\(\delta^\top H\delta\)、\(\nabla f^\top\delta\)、沿方向的二阶导数

**符号翻译**：

| 记号 | 含义 |
|---|---|
| \(\boldsymbol x=(x,y)^\top\) | 整体二维位置；非粗体 \(x\) 是第一坐标 |
| \(\boldsymbol\delta=(a,b)^\top\) | 从某位置移动的二维位移 |
| \(\boldsymbol d=(u,v)^\top\) | 方向向量 |
| \(t\) | 沿方向前进的标量 |
| \(\nabla f\) | 各一阶偏导组成的列向量 |
| \(H_f\) | 各二阶偏导组成的 Hessian 矩阵 |
| \({}^\top\) | 转置：行与列互换 |
| \(d^2/dt^2\) | 对 \(t\) 求两次导数 |
| \(\big|_{t=0}\) | 导数求出后令 \(t=0\) |

**1. 二阶泰勒式的矩阵乘法**：

给定 \(f(x,y)=x^2-y^2\)，则 \(\nabla f(\boldsymbol0)=\boldsymbol0\) 且 \(H=\begin{pmatrix}2&0\\0&-2\end{pmatrix}\)。令 \(\boldsymbol\delta=(a,b)^\top\)，
\[
\delta^\top H\delta
=\begin{pmatrix}a&b\end{pmatrix}
\begin{pmatrix}2&0\\0&-2\end{pmatrix}
\begin{pmatrix}a\\b\end{pmatrix}
=\begin{pmatrix}a&b\end{pmatrix}
\begin{pmatrix}2a\\-2b\end{pmatrix}
=2a^2-2b^2.
\]
并且 \(\nabla f(\boldsymbol0)^\top\delta=(0,0)(a,b)^\top=0\)，所以
\[
f(\delta)=f(\boldsymbol0)+\nabla f(\boldsymbol0)^\top\delta+\frac12\delta^\top H\delta
=0+0+\frac12(2a^2-2b^2)=a^2-b^2.
\]
这对当前纯二次函数是精确等式。一般在 \(\boldsymbol0\) 附近的二阶展开有 \(o(\|\delta\|^2)\) 余项。公式中的 \(f(0)\) 应理解为 \(f(\boldsymbol0)\)，输入是零向量，不是普通的一元数值零。

**2. 沿方向的一元求导**：

令 \(\boldsymbol x=(a,b)^\top\)、\(\boldsymbol d=(u,v)^\top\)。那么
\[
\boldsymbol x+t\boldsymbol d=(a+tu,b+tv)^\top
\]
定义 \(g(t)=f(\boldsymbol x+t\boldsymbol d)=(a+tu)^2-(b+tv)^2\)，则
\[
g'(t)=2u(a+tu)-2v(b+tv),\qquad g''(t)=2u^2-2v^2.
\]
另一方面，
\[
d^\top Hd=(u,v)\begin{pmatrix}2&0\\0&-2\end{pmatrix}(u,v)^\top=2u^2-2v^2.
\]
因此 \(g''(0)=d^\top H_f(\boldsymbol x)d\)，是沿 \(d\) 方向出发时的二阶曲率，不是对 \(x\) 或 \(y\) 单独求导。沿 \(d=(1,0)\) 的曲率是 2；沿 \(d=(0,1)\) 的曲率是 -2。

**3. 两式联系**：若位移为 \(\delta=td\)，则
\[
\frac12\delta^\top H\delta=\frac{t^2}{2}d^\top Hd.
\]
前者是二阶变化量，后者是局部方向曲率。

**学习策略**：先看清每个符号是标量、向量、还是矩阵，写出维度，再用普通行列乘法展开成数字。

In [ ]:
# 小数值核验：按普通标量与矩阵两种方式算相同的数值
import numpy as np
delta = np.array([0.1, 0.2], dtype=float)
direction_x = np.array([1.0, 0.0])
direction_y = np.array([0.0, 1.0])
H_saddle = np.array([[2.0, 0.0], [0.0, -2.0]])
direct = saddle(delta)
taylor = 0.5 * (delta.T @ H_saddle @ delta)
x_curvature = direction_x.T @ H_saddle @ direction_x
y_curvature = direction_y.T @ H_saddle @ direction_y
print("f(delta)=", direct, "Taylor second-order=", taylor)
print("directional curvatures:", x_curvature, y_curvature)
assert np.isclose(direct, -0.03)
assert np.isclose(direct, taylor)
assert np.isclose(x_curvature, 2.)
assert np.isclose(y_curvature, -2.)

### 下一次理解检查（等待学生）

对于 \(f(x,y)=x^2-y^2\)，令 \(\delta=(0.1,0.2)^\top\) 和 \(H=\operatorname{diag}(2,-2)\)。请学生独立计算：
1. \(\delta^\top H\delta\)；
2. \(\frac12\delta^\top H\delta\)；
3. \(f(0.1,0.2)\)。

把二阶矩阵写成标量乘法。验收后继续官方 Exercise 3 对比三优化器；目前课程 01/08 尚未完成。

## 二次型计算理解检查：学生独立答案（2026-11-09）

> \[
> \boldsymbol\delta^\top H\boldsymbol\delta
> \] = -0.06
>
> \[
> \frac12\boldsymbol\delta^\top H\boldsymbol\delta
> \] = -0.03
>
> \(f(0.1,0.2)=0.1^2-0.2^2\)=-0.03


**验收：3/3 完全正确。**

令 \(\delta=(0.1,0.2)^\top\)，\(H=\begin{pmatrix}2&0\\0&-2\end{pmatrix}\)，有：
\[
H\delta=(0.2,-0.4)^\top,\quad
\delta^\top H\delta=0.1\cdot0.2+0.2\cdot(-0.4)=-0.06.
\]
所以 \(\frac12\delta^\top H\delta=-0.03=f(0.1,0.2)\)。学生已能把矩阵二次型与标量计算对应起来，符号基础理解检查通过。

**严谨补充：** 二次型数值不是 Hessian 的某个固定特征值。其符号与大小取决于位移在各特征方向的分解。对于本二次函数，在原点二阶泰勒无高阶余项。

**教学状态：** Hessian 符号补课结束。官方 Exercise 3 鞍点函数与梯度已实现，下一步做 GD、Momentum、Adam 实验；Exercise 3 还未完成，学习主进度不应提前标记。

### 下一步：官方 Exercise 3 比较三种优化器（待学生编程）

沿用此前学生已写的 \`saddle(params)\`、\`saddle_gradient(params)\`、\`optimize(optimizer,func,grad_func,start,steps)\`。使用：
- 初始位置 \`saddle_start=[0.01,0.01]\`，迭代预算 100 步；
- GD: \`GradientDescent(lr=0.01)\`；
- Momentum: \`SGDMomentum(lr=0.01,momentum=0.9)\`；
- Adam: \`Adam(lr=0.01)\`；
- 为三者分别创建新优化器，记录 \`history[-1]\` 与 \`saddle(history[-1])\`，比较 y 分量变化；
- 先提示学生填写 GD 的一次调用，其余尽量自己类比写，保持一问一答、学生独立实现。

**预先验算配置的数值稳定性（教师独立执行 Python float，结果不提前告诉学生）：**
- 100步 GD: approx [0.0013261955589475322, 0.07244646118252342], f ≈ -0.0052467309432103005；
- 100步 Momentum: approx [4.228113718128517e-05, 182.4778746602841], f ≈ -33298.174740532566；
- 100步 Adam: approx [-1.494326198489768e-05, 1.3820185899674133], f ≈ -1.9099753827922161.
数值发散在此目标中可能指朝无下界方向逃离鞍点，但负的 f 不代表得到有限最优值；切勿将最终损失最负误称泛化意义上的最优优化器。用用户实际运行日志验收再给出比较结论。

In [ ]:
saddle_start = [0.01, 0.01]
saddle_steps = 100

optimizers = {
    "GD": GradientDescent(lr=0.01),
    "Momentum": SGDMomentum(lr=0.01, momentum=0.9),
    "Adam": Adam(lr=0.01),
}

for name, optimizer in optimizers.items():
    history = optimize(
        optimizer,
        saddle,
        saddle_gradient,
        saddle_start,
        saddle_steps
    )

    final_params = history[-1]
    final_loss = saddle(history[-1])

    print(name, final_params, final_loss)

## 官方 Exercise 3：三优化器鞍点逃逸 — 学生代码验收（2026-11-09）

**原始代码已保存在上一单元。验收：正确。** 同一初始点 [0.01,0.01]、100 步、learning rate 0.01，Momentum 系数 0.9，三种优化器分别新建对象并调用学生自写的优化流程；最后取历史最后一个参数和对应 saddle 值。用户本轮**没有附实际 stdout**，以下是教师通过独立 Python 运行确认的数值，不声称是用户本地输出。

**独立运行的实现口径**：GD 用 p-lr*g；Momentum 用 v=.9*v+g、p=p-lr*v 且初始零速度；Adam 默认参数为 beta1=.9、beta2=.999、epsilon=1e-8，含一二阶 EMA 与偏差校正；解析梯度为 [2x,-2y]，原始 optimize 流程跑 100 步。

| optimizer | x at 100 | y at 100 | f(x,y) at 100 |
|---|---:|---:|---:|
| GD | 0.0013261955589475322 | 0.07244646118252342 | -0.0052467309432103005 |
| Momentum | 4.228113718128517e-05 | 182.4778746602841 | -33298.174740532566 |
| Adam | -1.4943261984896815e-05 | 1.3820185899674127 | -1.9099753827922143 |

三种方法在该起点下都沿 y 不稳定方向离开原点附近。不能把 Momentum 最负 Loss 判为通用最佳，因为 f=x^2-y^2 沿 |y| 趋于无穷时无下界。逃逸是离开驻点附近，不意味着找到有限极小值。

**补充可复验的逃逸速度**：设定判据 |y_t|>=0.05（初始 y_0=0.01），首达步数分别为 GD 82、Momentum 20、Adam 5；若换为 |y_t|>=0.1，GD 在 100 步内未达，Momentum 27、Adam 10。阈值可能改变相对比较，因此实验比较必须报告阈值。此阈值是用于诊断的补充定义，不是官方指定阈值。

**进度判定**：本轮官方 Exercise 3 的代码和数学结果已通过，但尚待学生针对为何关注 y 而非最终 Loss 解释。继续一问一答；整个 01/08 课程尚未完成。

In [ ]:
# 独立验证：以负曲率方向的首达阈值比较三种优化器
new_optimizers = {
    "GD": GradientDescent(lr=0.01),
    "Momentum": SGDMomentum(lr=0.01, momentum=0.9),
    "Adam": Adam(lr=0.01),
}
threshold = 0.05
for name, opt in new_optimizers.items():
    h = optimize(opt, saddle, saddle_gradient, [0.01, 0.01], 100)
    first_escape_step = next(
        (i for i, p in enumerate(h) if abs(p[1]) >= threshold),
        None
    )
    print(name, "final_params=", h[-1],
          "final_loss=", saddle(h[-1]),
          "first_step_|y|>=0.05=", first_escape_step)
# 独立 Python 运行：GD=82，Momentum=20，Adam=5。

### 下一轮理解检查（等待学生）

本题目标函数 $f(x,y)=x^2-y^2$ 沿 $|y|\to\infty$ 方向无下界。请解释：

**为什么判断三种优化器是否离开鞍点附近时，应关注 $|y_t|$ 的变化（或明确邻域的首达时间），而不能单凭最终 Loss 越负越好来排名？**

答完后结束官方 Exercise 3，继续官方 Exercise 4：指数学习率衰减 $\eta_t=\eta_0(0.999)^t$。

## 官方 Exercise 3：逃逸评价标准的理解检查——学生原始回答（2026-11-09）

> 因为在这个鞍点的实验中他们的y的梯度方向是一致的,所以momentum累积起来非常快,最终loss最低

### 教师验收：部分正确，需要补齐评价标准

学生解释对了 **Momentum 为什么让 y 迅速远离原点**，但还没有回答原题的关键：为何不能仅因第 100 步的最终 Loss 最低就认定 Momentum 是这三种方法中最好的。

对于 $f(x,y)=x^2-y^2$，$g_y=-2y$。在本实验从 $y_0=0.01>0$ 出发的这组轨迹中，y 始终为正，$g_y$ 为负；Momentum 的 $v_{y,t}=\beta v_{y,t-1}-2y_t$ 不断积累负值，更新 $y_{t+1}=y_t-\eta v_{y,t}$ 使 y 增大，因而 $-y^2$ 向负无穷方向下降。这里所谓“方向一致”仅针对该特定轨迹，不是鞍点实验的一般结论。

**缺失的判断标准**：$f(x,y)$ 沿 $|y|\to\infty$ 无下界，因而不存在有限的全局最小点；在固定 100 步预算下，Momentum 的最终 $f$ 最负只能说明它在这一时刻沿负曲率方向远离原点得更多。若指标是“最快逃离给定邻域”例如首达 $|y|\ge0.05$，此前独立实验给出 Adam 第 5 步、Momentum 第 20 步、GD 第 82 步；所以固定步数目标值、逃逸时间、稳定收敛是不同的问题，不能混为一谈。

**准确表述**：Momentum 第 100 步目标值最低是事实；根据该目标值给出普适“优化器最好”的结论不成立，尤其无法代表“逃逸最快”或“找到有限最优解”。

学生理解：动力学解释通过，研究问题与指标区分尚需再检查。Exercise 3 的实验代码通过，但概念验收尚未完全结束。

### 下一道精简理解题（等待学生）

假设实验把一个优化器的参数推进到了 $(x,y)=(0,10^6)$，函数值达到 $-10^{12}$。

请学生独立回答：这么负的函数值能证明它找到了最优点，或者最先逃离了 $|y|<0.05$ 的邻域吗？请用“无下界”和“首达步数”各解释一句。

验收通过后结束 Exercise 3，再进入官方 Exercise 4：GD 指数学习率衰减 $\eta_t=\eta_0\cdot0.999^t$，要求学生独立添加步数状态并在每次更新前计算学习率，和固定学习率的 GD 对比 Rosenbrock。

## 官方 Exercise 3：逃逸指标理解检查通过（2026-11-09）

**学生原始回答：**

> 不能,这个函数本身就没有最优点,也不能证明他先逃出邻域,要看那个优化器最先逃出邻域的步数

**验收：正确。Exercise 3 完成。** 函数 \(f(x,y)=x^2-y^2\) 沿 \(|y|\to\infty\) 没有下界，因而不存在有限全局最小值；最终目标值不能证明已找到最优点，也不能证明最快逃出指定邻域。要比较逃逸速度需查看每条轨迹达到阈值的首达步数，例如 \(\min\{t:|y_t|\ge 0.05\}\)。

官方 Exercise 1、2、3 已完成；Exercise 4 尚未完成，不更新整课 LEARNING.md 完成状态。

## 官方 Exercise 4：指数学习率衰减（开题）

定义：
\[
\eta_t=\eta_0\gamma^t,\qquad \gamma=0.999,\quad t=0,1,2,\ldots
\]
与固定学习率的 GD 作比较：
\[
\theta_{t+1}=\theta_t-\eta_t\nabla f(\theta_t).
\]

- \(\eta_0\)：初始学习率，例如 0.001。
- \(\gamma\)：单步衰减率（本题 0.999）。
- \(t\)：当前已完成的更新次数，从 0 开始。
- 第一次 step 时用 \(\eta_0\)，完成第一次参数更新后 t 变 1，第二次 step 用 \(\eta_0\cdot0.999\)。

**特别注意：** 先计算本次 current_lr、更新参数，再 t += 1，避免 off-by-one。对照实验要固定相同起点、初始学习率、迭代预算和损失函数。

### 下一步：学生独立实现指数衰减 GD

为避免改坏旧版 GradientDescent，先写独立类 DecayGradientDescent：

1. 初始化 lr0=0.001、gamma=0.999、t=0；
2. 在 step(params,grads) 里求 current_lr = ?；
3. 用列表推导式求更新后的 new_params = ?；
4. 最后 self.t += 1，再返回新参数。

等学生提交后先测试第一、二步更新公式，再用 Rosenbrock、起点 [-1,1]、5000 步、lr0=0.001，比较固定 GD 与指数衰减 GD 的最终 Loss 和损失曲线。正式 Exercise 4 原文要求向 GradientDescent 加入 schedule；教学时先以独立类隔离验证，后续再合并。

In [ ]:
class DecayGradientDescent:
    def __init__(self, lr0=0.001, gamma=0.999):
        self.lr0 = lr0
        self.gamma = gamma
        self.t = 0

    def step(self, params, grads):
        # 1. 计算当前学习率
        current_lr = self.lr0 * self.gamma ** self.t

        # 2. 更新参数
        new_params = [p - current_lr * g for p,g in zip(params,grads)]

        # 3. 完成一次更新后，步数加 1
        self.t += 1

        return new_params

## 官方 Exercise 4：指数学习率衰减实现验收（2026-11-09）

**学生理解检查原文：**
> 因为最先开始这个的衰减系数要用0次幂变为1,让他在第0步数的时候保持lr=lr0

**验收：代码完全正确，理解检查正确。** 学生独立实现了 DecayGradientDescent 类；每次先计算 current_lr=lr0*gamma**t，然后用列表推导式更新参数，再 t+=1。第一次更新使用原始学习率，第二次更新开始衰减，未出现 off-by-one 错误。

**教师独立 Python 执行验证：** 固定梯度 [0.1,-0.2]，初始参数 [1.0,2.0]，连续调用三次：

| 更新次数 | 计算时 t | 当次 lr | 更新后的参数 |
|---:|---:|---:|---|
| 1 | 0 | 0.001000000 | [0.9999, 2.0002] |
| 2 | 1 | 0.000999000 | [0.9998001, 2.0003998] |
| 3 | 2 | 0.000998001 | [0.9997002999, 2.0005994002] |

测试均通过；包含学习率、更新参数和计数检查。t=5000 时学习率约 6.7211e-6，初始的约 0.672%；这可能有利于后期细化，也可能令前期尚未收敛时停止取得进展。因此必须比较控制变量下的两种方案。

**课程进度：** 此处只完成独立类的实现与前三步验证。官方 Exercise 4 仍需要在 Rosenbrock 上用相同起点、初始 lr 和 5000 步预算做固定学习率与衰减学习率对照；最终还需要按官方要求将可选衰减集成回 GradientDescent 类。

In [ ]:
# 教师测试：原始类的前 3 步学习率与参数更新
decay_check = DecayGradientDescent(lr0=0.001, gamma=0.999)
params_check = [1.0, 2.0]
expected_lrs = [0.001, 0.000999, 0.000998001]
expected_params = [
    [0.9999, 2.0002],
    [0.9998001, 2.0003998],
    [0.9997002999, 2.0005994002]
]
import math
for i in range(3):
    actual_lr = decay_check.lr0 * decay_check.gamma ** decay_check.t
    assert math.isclose(actual_lr, expected_lrs[i], rel_tol=1e-12)
    params_check = decay_check.step(params_check, [0.1, -0.2])
    assert all(math.isclose(a,b,rel_tol=1e-12,abs_tol=1e-12) for a,b in zip(params_check,expected_params[i]))
    print(f"update={i+1}, lr={actual_lr:.9f}, params={params_check}")
assert decay_check.t == 3
print("PASS: decay schedule, updates, and step count")

### 下一步：学生独立完成固定 GD 与衰减 GD 的 Rosenbrock 对照

保持起点 start=[-1,1]、步数 5000 和相同初始学习率 lr0=0.001。分别用 GradientDescent(lr=0.001) 与 DecayGradientDescent(lr0=0.001,gamma=0.999)，调用此前实现的 optimize、rosenbrock 和 rosenbrock_gradient，计算两组最终 loss。

待学生提交代码与结果后再验收，随后制作对数纵轴的两组 loss 曲线，并讨论为何衰减学习率不总优于固定学习率。官方要求向 GradientDescent 加入指数学习率 schedule：当前是独立类教学阶段，还需将选项合并回原类。此课还要完成 Use It、Ship It、Post Quiz，未整体完成。

In [ ]:
fixed_history = optimize(
    GradientDescent(lr=0.001),
    rosenbrock,
    rosenbrock_gradient,
    start,
    steps
)

decay_history = optimize(
    DecayGradientDescent(lr0=0.001, gamma=0.999),
    rosenbrock,
    rosenbrock_gradient,
    start,
    steps
)

fixed_loss = rosenbrock(fixed_history[-1])
decay_loss = rosenbrock(decay_history[-1])

print("Fixed GD:", fixed_loss)
print("Decay GD:", decay_loss)

## 官方 Exercise 4：固定学习率 vs 指数衰减 GD — 学生代码验收（2026-11-09）

学生原始代码完整保留在上方代码单元。**验收：正确**，调用相同的 Rosenbrock、解析梯度、起点 [-1,1]、5000 步，初始学习率相同 (0.001)，指数衰减组 gamma=0.999，最终从历史最后一个参数计算 loss。

以下均为教师在独立 Python float 环境实际运行结果，**用户本轮没有附实际 stdout**：

| 方法 | 第 5000 步最终参数 | 第 5000 步最终 Loss |
|---|---|---:|
| Fixed GD | [0.9398260387603975, 0.8830233446769815] | 0.0036271375470905893 |
| Decay GD | [0.35776010988119367, 0.12494757410083225] | 0.4133991097394533 |

**教师实际取得的全程采样 Loss**：

| 步 t | Fixed Loss | Decay Loss |
|---:|---:|---:|
| 0 | 4.0 | 4.0 |
| 100 | 3.6647272288743595 | 3.680889369618588 |
| 500 | 2.0975703354208024 | 2.5676098710233037 |
| 1000 | 0.40587541600795535 | 1.4506762615306223 |
| 2000 | 0.07417589888289902 | 0.6048116733709038 |
| 3000 | 0.023826419883966183 | 0.4650053790175708 |
| 4000 | 0.00896305175393721 | 0.42632867464065294 |
| 5000 | 0.0036271375470905893 | 0.4133991097394533 |

本次 schedule 在第 5000 次参数更新时采用 eta_4999=6.727839799665254e-6，5000 步运行结束后的 eta_5000=6.721111959865589e-6。固定 GD 在该实验预算下明显胜出，原因之一是衰减 GD 的步长过早降得很小，尚未接近 (1,1) 即显著放缓；不能据此推断指数衰减普遍无效。

官方 Exercise 4 要求给 GradientDescent 类加入指数衰减 schedule。当前用户已独立完成 DecayGradientDescent 类和对照实验，但仍要进行结果口头解释、把功能合并到原 GD 类。整课尚有 Use It、Ship It、Post Quiz。

In [ ]:
# 教师可复核检查（运行前已执行上方学生代码）
import math
assert len(fixed_history) == len(decay_history) == 5001
assert math.isclose(rosenbrock(fixed_history[-1]), 0.0036271375470905893, rel_tol=1e-8)
assert math.isclose(rosenbrock(decay_history[-1]), 0.4133991097394533, rel_tol=1e-8)
for t in (0, 100, 500, 1000, 2000, 3000, 4000, 5000):
    print(t, rosenbrock(fixed_history[t]), rosenbrock(decay_history[t]))
print("LR of 5000th update:", 0.001 * 0.999**4999)
print("LR after 5000 updates:", 0.001 * 0.999**5000)
print("PASS: fixed/decay GD")

### 下一道理解检查（等待用户回答）

这次 Decay GD 的第 5000 步最终 Loss 更高，参数也仍远离 Rosenbrock 最优点 (1,1)。请结合指数式 \(\eta_t=\eta_0\cdot0.999^t\) 解释产生差异的主要原因，并判断是否能够推广为“指数衰减策略一定不好”。

回答后让学生将可选学习率衰减 schedule 整合到原 GradientDescent 类并做回归测试；再进入 Use It、Ship It、Post Quiz。

## 官方 Exercise 4：指数衰减对照实验理解检查（2026-11-09）

**学生原始回答：**

> 因为我们还没有到达最优解附近学习率就已经衰减的很小了,但是这也不能证明指数衰减策略不好,因为这个要考虑考虑目标函数、初始学习率、衰减速度与训练预算之间的关系。

### 教师验收：通过

学生正确解释了固定 GD 与衰减 GD 的 5000 步对照实验，并明确不能从某一个 Rosenbrock 实验推广出指数衰减普遍无效。指数学习率 \(\eta_t=\eta_0\gamma^t\) 在 \(\eta_0=0.001,\gamma=0.999,t=5000\) 时约为 \(6.72\times10^{-6}\)，即初始的 0.672%。在先前独立实际复现实验中，固定 GD 最终 Loss 为 0.0036271375470905893，衰减 GD 为 0.4133991097394533。原因是本次衰减速度与优化进度、训练预算不匹配；调整 \(\gamma\)、预算或初始化可能改变结果。可以进一步试 \(\gamma=0.9999\)，但在实际运行之前不宣称一定更好。

官方 Exercise 4 的“固定 vs 衰减对照”和理解检查已经完成；还需按官方题目把指数调度选项整合回 GradientDescent（旧类接口保持可用），才能给 Exercise 4 代码的最终验收。整课后续还有 Use It、Ship It、Post Quiz；未完成整课。

### 下一步：修改原 GradientDescent 支持可选衰减（等待学生独立编码）

采用最简洁且向后兼容的接口：
\`\`\`python
class GradientDescent:
    def __init__(self, lr=0.001, gamma=1.0):
        self.lr = lr
        self.gamma = gamma
        self.t = 0

    def step(self, params, grads):
        current_lr = ???
        new_params = ???
        self.t += 1
        return new_params
\`\`\`
其中 gamma=1.0 表示原来的固定学习率 GD；gamma=0.999 表示指数衰减 GD。强调 self.t 从 0 开始，计算当步学习率之后再增长。用户独立完成后进行回归测试：固定模式的前三步与旧 GD 相同，衰减模式的前三步与此前 DecayGradientDescent 相同；再用 Rosenbrock 5000 步与独立类对照，防止破坏既有行为。完成后进入 Use It、Ship It 和 Post Quiz。

In [ ]:
class GradientDescent:
    def __init__(self, lr=0.001, gamma=1.0):
        self.lr = lr
        self.gamma = gamma
        self.t = 0

    def step(self, params, grads):
        # 1. 当前学习率
        current_lr = self.lr * self.gamma ** self.t

        # 2. 更新参数
        new_params = [p - current_lr * g for p,g in zip(params,grads)]

        # 3. 记录已完成的更新次数
        self.t += 1

        return new_params

## 官方 Exercise 4：整合指数学习率衰减至原 GD（2026-11-09）

原始学生提交的完整类已保存在上一代码单元。判定：通过。默认 gamma=1.0 保留固定学习率；gamma=0.999 时按 eta_t=lr*gamma^t 衰减。用户先计算当步学习率并更新，再 t+=1，没有步数偏移。

**教师在独立 Python 实际运行核验：**
- 初始参数 [1,2]、梯度 [0.1,-0.2]，gamma=1.0 连续三步的参数依次为 [0.9999,2.0002]、[0.9998,2.0004]、[0.9997,2.0006]；
- gamma=0.999 连续三步依次为 [0.9999,2.0002]、[0.9998001,2.0003998]、[0.9997002999,2.0005994002]；
- 两种配置的更新后三步 t=3，且与先前独立 DecayGradientDescent 类对应的结果逐项相同。
- Rosenbrock [-1,1]、5000 步、初始 lr=0.001：固定方案最终 Loss 0.0036271375470905893；衰减方案最终 Loss 0.4133991097394533。两个方案的整条历史轨迹分别与对应独立类逐项相同。

**官方 Exercise 4 完成，Exercises 1—4 全部完成。** 后续仍有 Use It（PyTorch 现成优化器）、Ship It 和 post quiz，故根目录 LEARNING.md 暂不标记整课完成。

In [ ]:
# 教师可复核测试：合并后的 GD vs 独立 DecayGradientDescent
import math
for gamma in [1.0, 0.999]:
    merged = GradientDescent(lr=0.001, gamma=gamma)
    original_decay = DecayGradientDescent(lr0=0.001, gamma=gamma)
    p = [1.0, 2.0]
    q = [1.0, 2.0]
    for _ in range(3):
        p = merged.step(p, [0.1, -0.2])
        q = original_decay.step(q, [0.1, -0.2])
        assert p == q
    assert merged.t == original_decay.t == 3
    new_history = optimize(GradientDescent(lr=0.001, gamma=gamma), rosenbrock, rosenbrock_gradient, [-1.0,1.0], 5000)
    old_history = optimize(DecayGradientDescent(lr0=0.001, gamma=gamma), rosenbrock, rosenbrock_gradient, [-1.0,1.0], 5000)
    assert new_history == old_history
    print(gamma, rosenbrock(new_history[-1]))
print("PASS: old and new schedules match step-for-step")

### 下一步：官方 Use It（等待学生）

官方 Use It 学习如何调用 torch.optim.SGD、torch.optim.Adam、torch.optim.AdamW，以及 torch.optim.lr_scheduler.CosineAnnealingLR。先练习典型训练迭代中的 optimizer.zero_grad()、loss.backward()、optimizer.step()、scheduler.step()；区分 optimizer 负责更新参数、scheduler 负责调整未来的 learning rate。教师要提出一项简短理解检查，再让学生独立实现可运行的 PyTorch 片段。随后执行官方 Ship It 和 post quiz。

## 官方 Use It：PyTorch 的 backward 与 step 调用顺序（2026-11-09）

**学生原始回答：**
> 因为我们要先获得梯度再更新,所以loss.backward()在前,如果交换这两行的话,我们上来更新会不边,再第二个阶段再更新,相当于step()少了一次

### 验收：前半正确，后半修正

学生理解了先计算梯度再更新，但认为 step 和 backward 交换后只少更新一次。对标准循环：

    optimizer.zero_grad()
    loss = ...
    optimizer.step()       # 错序
    loss.backward()

如果使用 PyTorch 默认 zero_grad()（默认 set_to_none=True），在 step 执行前参数的 grad 为 None，SGD 不进行更新；当前 backward 的梯度在下一轮开始时被 zero_grad 清除。如此反复，参数可能永远不更新，而不只是滞后一轮。若不执行 zero_grad 且存在旧梯度，step-before-backward 也可能使用上一轮旧梯度，属于另一种训练语义；不能笼统说“总少一步”。

**教师独立实际执行 PyTorch 2.10.0+cpu**（参数 w=1，损失 w²，SGD lr=.1，重复三轮，分别比较标准顺序和错误顺序；均在每轮开头 zero_grad）：

| 轮次 | 正确：zero→backward→step | 错误：zero→step→backward |
|---|---:|---:|
| 0 | 1.0 | 1.0 |
| 1 | 0.800000011920929 | 1.0 |
| 2 | 0.6399999856948853 | 1.0 |
| 3 | 0.5119999647140503 | 1.0 |

这是一项真实运行的 PyTorch 行为验证，不是人工推断。需要理解三步对应关系：清除旧梯度→根据 loss 反向传播计算当前梯度→使用当前梯度更新参数。

In [ ]:
# 教师实跑的 PyTorch 2.10.0 CPU 验证：同一数学问题，不同调用顺序
import torch
for order in ("correct", "swapped"):
    w = torch.nn.Parameter(torch.tensor(1.0))
    optimizer = torch.optim.SGD([w], lr=0.1)
    values = [w.item()]
    for epoch in range(3):
        optimizer.zero_grad()
        loss = w.square()
        if order == "correct":
            loss.backward()
            optimizer.step()
        else:
            optimizer.step()
            loss.backward()
        values.append(w.item())
    print(order, values)
    if order == "correct":
        assert abs(values[1] - 0.8) < 1e-6
        assert abs(values[2] - 0.64) < 1e-6
        assert abs(values[3] - 0.512) < 1e-6
    else:
        assert values == [1.0, 1.0, 1.0, 1.0]
print("PASS: no useful update if step precedes backward in this standard zero_grad loop")

### 下一步小练习：使用 PyTorch 训练一个标量（待学生独立实现）

目标是令可训练标量 w 从 1.0 开始，最小化 L(w)=w²。给骨架：

    import torch
    w = torch.nn.Parameter(torch.tensor(1.0))
    optimizer = torch.optim.SGD([w], lr=0.1)
    for i in range(3):
        optimizer.zero_grad()
        loss = ???              # scalar differentiable loss
        ???                     # compute gradients
        ???                     # update parameter
        print(w.item())

学生亲自填写；预期结果来自教师实际运行 [0.8, 0.64, 0.512] (float32 有细小误差)。完成后介绍 AdamW 和 CosineAnnealingLR 的实际调用，以及 scheduler.step() 与 optimizer.step() 的关系。官方 Use It 后才进入 Ship It 和 Post Quiz。

**当前状态**：四道官方 Exercises 完成；Use It 进行中。不要把整个 01/08 课程标记为完成。

In [ ]:
import torch

w = torch.nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([w], lr=0.1)

for i in range(3):
    optimizer.zero_grad()

    loss = w ** 2

    # 计算梯度
    loss.backward()

    # 更新参数
    optimizer.step()

    print(f"step={i+1}, w={w.item():.6f}")

## 官方 Use It：PyTorch SGD 标量训练 — 学生代码验收（2026-11-09）

**验收：完全正确。** 学生独立补全 torch.nn.Parameter、SGD、zero_grad()、loss=w**2、loss.backward()、optimizer.step()、打印参数的新值。对应梯度 dL/dw=2w，更新规律 w_(t+1)=(1-2eta) w_t=0.8 w_t (eta=0.1)。

**教师实际运行环境：** PyTorch 2.10.0+cpu，原始学生代码真实 stdout：

    step=1, w=0.800000
    step=2, w=0.640000
    step=3, w=0.512000

执行过程中，step 调用之前的参数与梯度为：

| update | w_before | grad_before_step | loss_before_step | w_after |
|---|---:|---:|---:|---:|
| 1 | 1.0 | 2.0 | 1.0 | 0.800000011920929 |
| 2 | 0.800000011920929 | 1.600000023841858 | 0.64000004529953 | 0.6399999856948853 |
| 3 | 0.6399999856948853 | 1.2799999713897705 | 0.40959998965263367 | 0.5119999647140503 |

学生本轮未附 stdout；表和三行 stdout 是教师独立运行学生的原始代码，不冒充用户本机日志。说明了 w.item() 读取标量、PyTorch 参数梯度默认累积并且应 zero_grad() 的常规训练方式。

**进度：** Use It 的 SGD 自动求导/参数更新小练习通过；下一步 AdamW 和学习率调度器 CosineAnnealingLR 及 scheduler.step() 的角色。整课仍待 Ship It 与 post quiz，不更新 LEARNING.md。

In [ ]:
# 教师实际复核（PyTorch 2.10.0 CPU）
import torch
w = torch.nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([w], lr=0.1)
expected = [0.8, 0.64, 0.512]
for i in range(3):
    optimizer.zero_grad()
    loss = w**2
    loss.backward()
    optimizer.step()
    assert abs(w.item() - expected[i]) < 1e-6
    print(f"step={i+1}, w={w.item():.6f}")
print("PASS: automatic differentiation and SGD updates")

### 下一项 Use It 实操：AdamW + CosineAnnealingLR（等待学生）

先说明 PyTorch 里的 torch.optim.AdamW(model.parameters(),lr=...,weight_decay=...) 管理参数更新，torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=...) 管理学习率调度。scheduler.step() 通常在 optimizer.step() 之后，并且真实训练通常按 epoch 或计划步数调用；设置 T_max 要与调用次数单位一致。

给学生骨架：w=torch.nn.Parameter(torch.tensor(1.0))，AdamW([w],lr=0.01,weight_decay=0.01)，CosineAnnealingLR(optimizer,T_max=3)。三步执行 zero_grad，loss=w**2，backward，optimizer.step，scheduler.step，每步打印使用前/更新后的 lr（optimizer.param_groups[0]["lr"]）。让学生自行补 optimizer、scheduler 和核心操作，不提前提供所有答案。

**教师独立 PyTorch 验证：**
- 第 1 步 lr_used=0.01，scheduler 后 lr_next=0.0075；
- 第 2 步 lr_used=0.0075，lr_next=0.0025；
- 第 3 步 lr_used=0.0025，lr_next=0；
- w 最后约 0.979804933（float32）。
要明确日志中 lr_next 是下一次 optimizer.step 的学习率，而不是刚刚完成的本次更新所用学习率。

In [ ]:
import torch

w = torch.nn.Parameter(torch.tensor(1.0))

optimizer = torch.optim.AdamW([w],lr=0.01,weight_decay=0.01)  # AdamW, lr=0.01, weight_decay=0.01
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=3)  # CosineAnnealingLR, T_max=3

for i in range(3):
    optimizer.zero_grad()

    loss = w ** 2
    loss.backward()

    lr_used = optimizer.param_groups[0]["lr"]

    optimizer.step()

    scheduler.step()  # 更新学习率调度器

    lr_next = optimizer.param_groups[0]["lr"]

    print(
        f"step={i+1}, "
        f"w={w.item():.6f}, "
        f"used_lr={lr_used:.6f}, "
        f"next_lr={lr_next:.6f}"
    )

## Use It：AdamW + CosineAnnealingLR 学生实操（2026-11-09）

学生原始理解回答：

> 因为我们通过余弦退火要调整学习率,改变的是要调整的学习率,我们是不是少讲了一部分内容,这一节课

**验收：代码正确，回答正确但需把当次学习率与下一次学习率区别讲清楚。**

教师实际执行了完整原始代码，使用 PyTorch 2.10.0+cpu：

    step=1, w=0.989900, used_lr=0.010000, next_lr=0.007500
    step=2, w=0.982328, used_lr=0.007500, next_lr=0.002500
    step=3, w=0.979805, used_lr=0.002500, next_lr=0.000000

在首次退火区间且 eta_min=0、eta0=0.01、T_max=3 时：
\[
\eta_t=\frac{\eta_0}{2}(1+\cos(\pi t/T_{\max})).
\]
t=0,1,2,3 对应 0.01、0.0075、0.0025、0。scheduler.step() 修改的是 optimizer.param_groups 中将供下一次参数更新使用的学习率；本次已经执行的 optimizer.step() 不会被追溯改变。T_max 按 scheduler.step() 调用次数计算，不自动指定是 batch 还是 epoch。

## 官方 01/08 Optimization 课程覆盖审计与教学调整（2026-11-09）

用户明确提出“我们是不是少讲了一部分内容,这一节课”。教师本轮核对了官方英文课件全部章节、quiz.json、课程教学技能要求，以及原有 158 个 Notebook 单元。

**已完成、有用户代码或理解检查证据**：
- 手写 GD、Momentum、Adam，Rosenbrock 轨迹及四项官方练习；
- Batch GD、SGD、Mini-batch SGD 的区别及随机梯度无偏性条件已有专门答题记录（不能误报缺失）；
- 严格鞍点的 Hessian 与方向曲率特例；
- PyTorch SGD 训练步，以及本轮 AdamW + CosineAnnealingLR API 的运行。

**确有内容未系统覆盖**：
1. 凸与非凸的严格定义、驻点/局部最优/全局最优的联系；严格鞍点的特例不能代替全章定义。
2. 损失地形 Loss Landscape 的参数空间二维切片可视化；Loss-vs-step 曲线不是 loss landscape。
3. 学习率调度的 Step Decay、Cosine Annealing 公式、Warmup + Decay 原理；虽然手写过 Exponential Decay，但余弦尚未解释公式和调用尺度。
4. AdamW 解耦权重衰减和 weight_decay 的更新数学；此前仅完成 API 调用。
5. Use It 介绍的 gradient clipping、parameter groups 等实战技巧仍需必要说明。

**尚未进行**：官方 Ship It 优化器选择指南产物，以及 3 道 stage=post 正式 Quiz；不提前披露题目答案。不修改根目录 LEARNING.md 完成状态。

**补课顺序**：先从刚用到的余弦学习率与 T_max 入手，接着 Step Decay/Warmup，之后凸性及损失地形，最后 AdamW/权重衰减及梯度裁剪。完成后 Ship It 和 Post Quiz。用户独立回答、实际代码与验证逐次追加，不推翻前面已完成的练习。

**科学严谨备注**：官方课件关于凸优化和尖锐/平坦极小值泛化的说法不能不加条件地当作普遍定理。

### 下一道理解检查：余弦调度频率

在当前 eta0=0.01、T_max=3、eta_min=0 的实验中，首次衰减周期学习率依次为 0.01、0.0075、0.0025、0。

理解检查：如果将 scheduler.step() 从每个 mini-batch 执行改成每个 epoch 执行，而 T_max 仍是 3，学习率是在 3 个 mini-batch 还是 3 个 epoch 后达到本周期最低值？为什么？

学生回答后继续补齐 Step Decay 与 Warmup，再凸性、损失地形、AdamW 与 weight decay，最终 Ship It 和 Post Quiz。

## 用户反馈与教学规则调整（2026-11-09）

**用户本轮原始回答与反馈**：
> 三个epoch,因为我们这个是根据调用多少次scheduler.step()来看的,还有我感觉你完全都没有讲东西,你是不是要看一看项目要求,你要给自己定下规则要严格遵守我们定下的规则,你是否要更新一下记忆

**理解检查：通过。** T_max 按调用 scheduler.step() 的次数计量；每个 epoch 调一次，T_max=3 就在 3 个 epoch 结束后达到首段退火的 eta_min（相应调用位置）；每 batch 调一次则是 3 个 batch。

**对教学批评的处理**：阅读仓库 skills/learn/SKILL.md、learning-artifacts/cn/AGENTS.md、LEARNING.md、原始 docs/en.md、quiz.json、课程 README，确认旧课中有些知识仅给过 API 或名词，尚未形成有证明的理解。已经重新审计并规定：完整课程覆盖表、直觉到数学到数值到工程的深讲、学生主动代码与真实实验、严格区分案例与定理、每次记录与复核。

**持久化位置**：学习仓库 my-progress 的 learning-artifacts/cn/AGENTS.md 新增“教学质量硬性验收规范”，当前课 README 新增覆盖清单。此为仓库规则，非 ChatGPT 系统级记忆写入，不得混淆。

**当前学习进度**：前述四项官方 Exercises 已完成。PyTorch SGD/AdamW + Cosine 基础调用通过；需补学 Cosine、Step Decay、Warmup、Convex/Non-convex、Loss Landscape、AdamW 解耦 Weight Decay、参数组和梯度裁剪等；Ship It 与官方 Post Quiz 未完成。下一知识单元：调度器的数学与训练尺度，必须讲透原理与数值例子，不只是提出下一道填空。


## Warmup + Cosine Decay：学生理解检查（2026-11-10）

**学生原始回答**：
> warmup的持续步数应该设置为1000,然后cosine的持续步数应该设置为9000

**验收：正确。** 总预算 N=10000 次参数更新，Warmup W=1000，余下 Cosine 阶段 C=N-W=9000。前提是两个调度器都在每次 optimizer.step 后各推进一个调度步，并通过 SequentialLR 依次激活，而不是同时推进。

### 完整推导：阶段连接、相对步数与 PyTorch 实际接口

设 eta_max=0.01、eta_min=0、N=10000、W=1000、C=9000，t 表示**已经完成的参数更新次数**。为了避免 warmup 真正从 0 开始，设置初始因子 a=0.001。

线性 Warmup 选用：
\[
\eta_t=\eta_{\max}\left[a+(1-a)\frac{t}{W}\right],\quad 0\le t<W.
\]

Cosine Decay 在全局 t=W 处重新设置相对时间 s=t-W：
\[
\eta_t=\frac{\eta_{\max}}2\left[1+\cos\left(\pi\frac{t-W}{C}\right)\right],
\quad W\le t\le N.
\]

对 Warmup 连续延伸到 t=W：eta_max*(a+1-a)=eta_max；对 Cosine 在 t=W：eta_max*(1+cos 0)/2=eta_max，所以连接处学习率**连续**。在 t=N 时 cosine 得 eta_min=0；注意 eta_N 是**N 次更新完成之后的下一次预备 LR**，最后一次实际参数更新用的是 eta_(N-1)，通常并非恰好 0。

**PyTorch 组合方式**：
- torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.001, end_factor=1.0, total_iters=1000)
- torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=9000, eta_min=0)
- torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup,cosine], milestones=[1000])

第 1 次更新使用 eta_0=1e-5；第 1000 次更新使用 eta_999=0.00999001；第 1000 次调用 scheduler.step 之后 eta_1000=0.01，之后 Cosine 按局部进度 s=t-1000 递减。本例 T_max 以 scheduler.step 调用次数计量。若按 epoch 频率调用，则语义改成 epoch，必须同步换算里程碑。注意 torch.optim.lr_scheduler.CosineAnnealingLR 单独在 T_max 之后继续调用的行为不等同于任意时刻钳位在 eta_min；避免把早到最小值误理解成永远维持最小值。

以下代码记录了教师在 PyTorch 2.10.0+cpu 下真实运行的检验，保留完整精度、不插值。

In [ ]:
# 教师独立运行：10,000 次训练、Warmup + Cosine 阶段边界检查
import math
import torch
w = torch.nn.Parameter(torch.tensor(1.0, dtype=torch.float64))
optimizer = torch.optim.SGD([w], lr=0.01)
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.001, end_factor=1.0, total_iters=1000
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=9000, eta_min=0.0
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[1000]
)
assert math.isclose(optimizer.param_groups[0]["lr"], 1e-5, rel_tol=1e-12)
for i in range(10000):
    lr_used = optimizer.param_groups[0]["lr"]
    optimizer.zero_grad()
    loss = w.square()
    loss.backward()
    optimizer.step()
    scheduler.step()
    if i + 1 in (1, 2, 999, 1000, 1001, 5000, 9999, 10000):
        print(f"completed={i+1}, used_lr={lr_used:.12f}, next_lr={optimizer.param_groups[0]['lr']:.12f}")
assert math.isclose(optimizer.param_groups[0]["lr"], 0, abs_tol=1e-15)
print("PASS: warmup/cosine milestone 1000 and total 10000 scheduler calls")

### 下一道理解检查

如果不改变 Warmup 和 Cosine 两个调度器自身的配置，只把 SequentialLR 的 milestones=[1000] 误设为 milestones=[9000]，训练仍然只有 10000 次 optimizer.step() 和 scheduler.step()，那么 Cosine 阶段实际只会执行多少次？最终学习率是否能按计划降到 0？请用“里程碑切换”和“局部计数”解释。

本课概念修复还包括凸/非凸严格定义、损失地形可视化、AdamW 解耦权重衰减和生产训练参数组/梯度裁剪。保持课程“未完成”，不更新根目录 LEARNING.md。

## 学习率调度里程碑：学生原始回答（2026-11-10）

> cosin阶段实际只执行了1000次调度,因为在第1000步的时候才切换为cosine,降不到0,因为只更新了1000步,

### 教师验收：阶段长度和终值判断正确，切换时间有误

学生正确计算了 Cosine 实际只执行 1000 次，并判断最终 LR 降不到 0。但错误声称“在第 1000 步才切换”。具体配置为 SequentialLR(..., milestones=[9000])，所以在**全局第 9000 次** scheduler.step 调用之后切换；第 10000 次全局调用后 Cosine 已经过了 1000 个局部调度步。

若全局步数 t 是已完成的优化器更新数量，Cosine 的局部步数为 s=t-9000（只适用于 t>=9000）。预设 T_max=9000，eta_0=0.01, eta_min=0，退火阶段第一段公式：

\[
\eta(s)=\frac{0.01}{2}\left(1+\cos\frac{\pi s}{9000}\right)
\]

因此 eta(1000)=0.009698463103929543，仍接近初始 0.01，并不为零。

**教师当轮实际执行 PyTorch 2.10.0+cpu**, 10000 次优化器更新和调度调用，记录的更新后学习率如下：

| completed updates | used_lr | next_lr |
|---:|---:|---:|
| 1 | 0.000010000000000 | 0.000019990000000 |
| 999 | 0.009980020000000 | 0.009990010000000 |
| 1000 | 0.009990010000000 | 0.010000000000000 |
| 8999 | 0.010000000000000 | 0.010000000000000 |
| 9000 | 0.010000000000000 | 0.010000000000000 |
| 9001 | 0.010000000000000 | 0.009999999695383 |
| 9999 | 0.009699655834367 | 0.009699059755432 |
| 10000 | 0.009699059755432 | 0.009698463103930 |

核心区分：全局训练步数 t 与局部调度步数 s；阶段切换要对准预期预算。数学与代码结果一致。

In [ ]:
# 教师可复核错误 milestone 实验：实际执行时 PyTorch 版本为 2.10.0+cpu
import torch
import math
w = torch.nn.Parameter(torch.tensor(1.0, dtype=torch.float64))
optimizer = torch.optim.SGD([w], lr=0.01)
warmup = torch.optim.lr_scheduler.LinearLR(optimizer,
        start_factor=0.001, end_factor=1.0, total_iters=1000)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=9000, eta_min=0.0)
schedule = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup,cosine], milestones=[9000])
for i in range(10000):
    lr_used = optimizer.param_groups[0]["lr"]
    optimizer.zero_grad()
    w.square().backward()
    optimizer.step()
    schedule.step()
    if i+1 in [1000, 8999, 9000, 9001, 10000]:
        print(i+1, lr_used, optimizer.param_groups[0]["lr"])
expected = 0.005*(1+math.cos(math.pi*1000/9000))
assert math.isclose(optimizer.param_groups[0]["lr"], expected, rel_tol=1e-11)
print("PASS:", expected)

## 待讲下一个官方 Concept：Convex vs Non-convex

从当前课程官方 docs/en.md 第 157 行的 Convex vs non-convex 转入。教师先给出数学定义（定义域为凸集；任意 x,y 和 lambda∈[0,1]），解释 Jensen 弦线不等式：
\[
f((1-\lambda)x+\lambda y)\le (1-\lambda)f(x)+\lambda f(y).
\]
然后说明局部极小点为什么在凸函数上必然全局极小，以及可微凸函数的驻点与全局最优的关系；不要直接把“梯度下降总能找到全局最优”当成无条件定理。

建议具体例子：h(z)=(z^2-1)^2，h(-1)=h(1)=0，但 h(0)=1，因此非凸，且 h'(0)=0、h''(0)=-4，证明“驻点未必是极小值”；Rosenbrock 的 Hessian H=[[2-400y+1200x²,-400x],[-400x,200]]，在 (0,1) 点为 diag(-398,200)，所以 Rosenbrock 在全 R² 不凸。注意本例即使非凸，其 (1,1) 仍为全局最小点 0。教师应引导学生亲自验证一个 Jensen 数值不等式，再进入 Loss Landscape；不要只给题不讲原理。

## 官方 Concept：Convex vs Non-convex 的严格数学含义（2026-11-10）

### 直观含义与严谨定义

形象而言，凸函数的图像在任意两点连接线的下方或连接线上；非凸函数至少有一处违反这一性质。这里的“弦线在函数上方”是全局性质，而非“看起来只有一个谷底”。

设定义域 \(D\subseteq\mathbb R^d\) 是**凸集**（对任意两点 x,y 及 lambda∈[0,1]，线段上的 (1-lambda)x+lambda y 也位于 D）。函数 \(f:D\to\mathbb R\) 称为凸，当且仅当对任意 x,y∈D 和 lambda∈[0,1]：

\[
f((1-\lambda)x+\lambda y)
\le (1-\lambda)f(x)+\lambda f(y).
\]

左边是参数线段内部的真实函数值；右边是两端函数值的线性插值。

### 为什么凸性与最优解有关？

如果 z 是凸函数的局部最小点，却存在 y 满足 f(y)<f(z)，则对足够小的 lambda>0，z+lambda(y-z) 仍在 z 的局部邻域内，由凸性：

\[
f(z+\lambda(y-z))
\le (1-\lambda)f(z)+\lambda f(y)<f(z),
\]

与 z 是局部最小点矛盾。因此凸函数任何局部最小点都是全局最小点。注意这并不自动保证**存在**最小点、**唯一**最小点或任意学习率下 GD **必然收敛**。

若 f 在凸开域上二次连续可微，凸性等价于所有点上的 Hessian 半正定 \(v^\top\nabla^2f(x)v\ge0\)（任意方向 v）。若可微凸函数存在无约束驻点 \(\nabla f(z)=0\)，一阶凸性不等式保证 z 是全局最小点。非凸函数在驻点的梯度也可能为零，但点可以是局部最大或鞍点。

### 从真实 Rosenbrock 测试函数推导一个反例

\[
R(x,y)=(1-x)^2+100(y-x^2)^2.
\]

两项均非负，在 (1,1) 处皆为 0，故其**全局最小值**是 0，且参数解 (1,1) 唯一。尽管如此，R **并不全局凸**。

其 Hessian 为

\[
\nabla^2R(x,y)=
\begin{pmatrix}
2-400y+1200x^2 &-400x\\
-400x&200
\end{pmatrix}.
\]

在 (0,1) 处 Hessian=diag(-398,200)，包含负特征值，证明它不可能在整个 \(\mathbb R^2\) 上是凸函数（Hessian 必须处处半正定）。

也可以直接验证 Jensen 不等式失败：取 a=(-0.1,1)、b=(0.1,1)，中点 m=(0,1)。
- R(a)=99.22
- R(b)=98.82
- R(m)=101
- 两端均值 99.02；凸性应要求 R(m)<=99.02，而实际 101>99.02，故非凸。

**重要区分**：“非凸”不意味着一定存在多个局部极小值，也不意味着一定找不到全局最优；Rosenbrock 就是一个具有唯一已知全局最小值 0、却并非整体凸的函数。

In [ ]:
# 教师独立 Python/NumPy 真实计算：Rosenbrock 非凸性两个证据
import numpy as np

def rosenbrock_hessian(x, y):
    return np.array([[2-400*y+1200*x*x, -400*x], [-400*x, 200.0]])

H = rosenbrock_hessian(0.0, 1.0)
assert np.allclose(H, np.diag([-398.0, 200.0]))
assert np.linalg.eigvalsh(H)[0] < 0

left = rosenbrock([-0.1,1.0])
right = rosenbrock([0.1,1.0])
middle = rosenbrock([0.0,1.0])
assert middle > 0.5*(left+right)
print("H(0,1) eigenvalues:", np.linalg.eigvalsh(H))
print("Jensen values:", left, middle, right, "mean endpoints:", 0.5*(left+right))
print("PASS: Hessian and Jensen violations show non-convexity")

### 独立理解检查：驻点不等于最小值

考虑一元函数 \(g(x)=(x^2-1)^2\)。请学生计算：
1. \(g'(0)\)；
2. \(g(0),g(1),g(-1)\)；
3. 判断 x=0 是否是最小值点；是否可以用这三个函数值证明 g 不是凸函数？

等待学生答题后逐项验收，不在题目后预先写出答案。若学生不熟悉 Jensen 中点形式，先使用已解释的 lambda=1/2 数字例子辅助，不要跳进下一章。

## 官方 Concept — Convex vs Non-convex：学生独立解题（2026-11-10）

**原始提交（逐字保留）：**

> \(g'(0)\) = 0 g(-1) = 0 g(0) = 1 g(1) = 0 x=0不是最小值点  g((1-0.5) -1 + 0.5 * 1) = g(0) = 1 > (1-0.5) * f(-1) + 0.5 * f(1) = 0

### 验收：全部正确，末尾 f 改为 g 是记号笔误

对 \(g(x)=(x^2-1)^2\)，利用链式法则：
\[
g'(x)=2(x^2-1)\cdot 2x=4x(x^2-1),\qquad g'(0)=0.
\]
函数值 \(g(-1)=g(1)=0,\ g(0)=1\)，且 \(g(x)\ge0\)，所以两点 \(-1,1\) 为全局最小点，\(x=0\) 不是。进一步 \(g''(x)=12x^2-4\)，故 \(g''(0)=-4<0\)，零点实际是**严格局部最大点**（此处二阶充分条件适用）。说明驻点与最优点并不等价。

用户正确应用 Jensen 的中点形式，取两端 \(x=-1,y=1,\lambda=1/2\)：
\[
g(0)=1>\frac{g(-1)+g(1)}2=0,
\]
违反凸性不等式，严格证明 \(g\) 在 \(\mathbb R\) 上非凸。一处反例足以反驳一个要求对所有 \(x,y,\lambda\) 都成立的定义。“f(-1),f(1)”属记号笔误，不影响证明。

**学情记录**：能独立求导、区分驻点与最小点、使用 Jensen 反例证明非凸性，故完成 Convex vs Non-convex 的本轮知识检查。以后仍可复习凸性局部最优即全局最优的证明与应用条件。

## 官方 Concept：Loss Landscape Visualization（损失地形）

### 真实问题与定义

优化时不是只看损失值随训练轮数的折线 \(L(\theta_t)\)；损失地形是把参数点 \(\theta\) 映射到训练损失 \(L(\theta)\)，考察参数空间中的高低、谷地、山脊和曲率。二者区别：
- loss-vs-step：横轴 t 是训练时间，纵轴是沿某条真实优化轨迹的 L；
- loss landscape：横轴/纵轴是参数方向，颜色或高度才是 L（或明确标注的单调变换）。

对于我们已经使用的 Rosenbrock，完整参数空间只有二维，等高线可以直接绘制 \(R(x,y)=(1-x)^2+100(y-x^2)^2\)；图中用 \(\log_{10}(1+R)\) 显示颜色，仅为压缩数值动态范围，并未改变 R 本身的最优点。

当神经网络参数为 \(\theta\in\mathbb R^d\)、d 极大时，无法直接画完整参数空间。一个常用二维切片是
\[
F(a,b)=L(\theta_{\mathrm{ref}}+a\,d_1+b\,d_2),
\]
其中 \(\theta_{\mathrm{ref}}\in\mathbb R^d\) 是参考权重，\(d_1,d_2\in\mathbb R^d\) 是两个给定方向（可以归一化，但必须说清口径），a、b 是切片坐标。画的是切片 F(a,b) 而非整个 L 的全局几何图景；切片中看不到的方向仍可能具有负曲率、其他极小值或高障碍。

### 从 Rosenbrock 的窄谷理解

\[
R(x,y)=(1-x)^2+100(y-x^2)^2.
\]
对于给定 x，将 y 设成 x² 会使第二项为零，因此 \(y=x^2\) 给出沿固定 x 截线上的最佳 y（弯曲谷底线），但**不是每个曲线上的点都在二维参数空间中满足梯度为零**。它还要沿 x 方向逐步滑向全局最小点 (1,1)。

这里区分“对 y 条件最优（\(\partial R/\partial y=0\)）”与“全参数驻点（\(\nabla R=0\)）”。这也是窄谷优化时方向耦合的直观来源。

### 科学限制

等高线显示的是函数在选定窗口和方向上的几何。不能凭单幅二维切片证明某个高维网络的全局最优唯一，也不能仅凭视觉上“宽/窄”就无条件断言泛化质量好坏；曲率、参数化方式、归一化和实际测试指标都可能影响判断。

In [ ]:
# 教师绘制并实际执行：真实 Rosenbrock 参数空间等高线，便于对比 loss-vs-step
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(-1.8, 1.8, 360)
ys = np.linspace(-0.6, 2.4, 300)
X, Y = np.meshgrid(xs, ys)
R = (1-X)**2 + 100*(Y-X**2)**2

fig, ax = plt.subplots(figsize=(8, 5.4))
contour = ax.contourf(X, Y, np.log10(1+R), levels=26)
fig.colorbar(contour, ax=ax, label="log10(1 + Rosenbrock loss)")
ax.contour(X, Y, np.log10(1+R), levels=10, linewidths=0.6, alpha=0.55)
vx = np.linspace(-1.5, 1.5, 200)
ax.plot(vx, vx**2, "--", label="Curved valley: y=x^2")
ax.plot([-1],[1],"o",markersize=9,label="Start (-1,1)")
ax.plot([1],[1],"*",markersize=14,label="Global minimum (1,1)")
ax.set(xlabel="Parameter x", ylabel="Parameter y",
       title="Rosenbrock loss landscape: contour view")
ax.set_xlim(-1.8,1.8)
ax.set_ylim(-0.6,2.4)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

assert float((1-1)**2 + 100*(1-1**2)**2) == 0
assert float((1-(-1))**2 + 100*(1-(-1)**2)**2) == 4

### 下一道理解检查：谷底曲线上的点是否全为驻点？

对 Rosenbrock \(R(x,y)=(1-x)^2+100(y-x^2)^2\)，给出已推导的解析梯度：
\[
\frac{\partial R}{\partial x}=-2(1-x)-400x(y-x^2),\qquad
\frac{\partial R}{\partial y}=200(y-x^2).
\]

**学生独立回答**：点 A=(-1,1) 位于曲线 y=x² 上，因此沿固定 x 的纵向截线是最低的；然而在完整二维优化问题上，这个点是不是驻点？请计算两项偏导并判断 \(\nabla R(A)\) 是否为零，解释“谷底线”与“驻点”的不同。

用户答题后记录真实计算并验收。然后继续权重衰减 AdamW 的数学机制、参数组与梯度裁剪，以及 Ship It + 官方后测。

## Loss Landscape：Rosenbrock 起点 (-1,1) 梯度理解检查（2026-11-10）

**学生原始回答：**

> 两个值都是0 ,都是驻点,因为这个是鞍点,我们还要看x轴方向

### 验收：部分正确，梯度计算与驻点判断错误

原式 \(R(x,y)=(1-x)^2+100(y-x^2)^2\) 的梯度为

\[
\nabla R(x,y)=\begin{pmatrix}
-2(1-x)-400x(y-x^2)\\
200(y-x^2)
\end{pmatrix}.
\]

在 \((-1,1)\) 时 \(y-x^2=0\)，所以 y 偏导 \(R_y=0\)，但 x 偏导的第一项仍是 \(-2(1-(-1))=-4\)。因此

\[
\nabla R(-1,1)=(-4,0)^\top\ne(0,0)^\top.
\]

**结论**：(-1,1) 不是驻点，也不是经典无约束优化中梯度为零的鞍点；只能说在固定 x=-1 的 y 方向上，该点使第二项最小。沿谷底约束曲线 \(y=x^2\) 有 \(R(x,x^2)=(1-x)^2\)，从 x=-1 朝 x=1 移动仍可使 Loss 从 4 降至 0。

更强的局部曲率核查：Hessian

\[
H_R(x,y)=\begin{pmatrix}
2-400y+1200x^2 &-400x\\
-400x&200
\end{pmatrix}
\]

在 (-1,1) 等于 \(\begin{pmatrix}802&400\\400&200\end{pmatrix}\)。左上角主子式 802>0，行列式 400>0，故 Hessian 在此点正定；这也说明不能依据“某个偏导数为零”判定鞍点。**即使 Hessian 正定，如果梯度非零，也不能称该点为局部最小点。**

**课堂提醒**：此前已讨论过真正的鞍点 \(f(x,y)=x^2-y^2\) 在 (0,0) 处的梯度为零、Hessian 有正负特征值，与当前 Rosenbrock 的 (-1,1) 情况不同。

In [ ]:
# 教师可复核：(-1,1) 的真实梯度、Hessian 与首次 GD 更新
import numpy as np
p0 = [-1.0,1.0]
grad0 = rosenbrock_gradient(p0)
assert np.allclose(grad0, [-4.0,0.0])
H0 = np.array([[802.0,400.0],[400.0,200.0]])
assert np.all(np.linalg.eigvalsh(H0)>0.0)
loss0 = rosenbrock(p0)
p1 = GradientDescent(lr=0.001).step(p0,grad0)
loss1 = rosenbrock(p1)
assert np.allclose(p1,[-0.996,1.0])
assert loss1 < loss0
print("Gradient at start:",grad0)
print("Hessian eigenvalues:",np.linalg.eigvalsh(H0))
print("First GD update:",p1,"loss before/after:",loss0,loss1)
print("PASS: not a stationary point; descent continues")

### 下一步独立检查：第一步 GD 后的 y 方向为何重新变化？

已知 \(\nabla R(-1,1)=(-4,0)\)，从 \(\theta_0=(-1,1)\) 出发，使用固定学习率 \(\eta=0.001\) 做标准 GD。

1. 计算第一次更新后的 \((x_1,y_1)\)；
2. 判断下一次计算梯度时 \(\partial R/\partial y(x_1,y_1)\) 是否仍然为 0；说明为什么。

教师已经独立复核过参数更新和后续 Loss，但**不要提前给出学生应答的梯度数值**。本次将原回答与纠正作为独立记录，尚未认为用户已掌握“谷底线 vs 驻点”的区别；等待下一次回答。

## Loss Landscape：第一次 GD 更新后梯度的学生独立回答（2026-11-10）

**原始回答：**

> (-0.996.1) y方向的梯度不是零了 y方向的梯度是1.5968

**验收：正确。** 把 (-0.996.1) 理解为输入标点笔误，意图是 (-0.996, 1)。教师不改写原始提交，只在讲评中说明。

### 详细订正与几何意义

Rosenbrock：

\[
R(x,y)=(1-x)^2+100(y-x^2)^2,
\quad
\nabla R=(-2(1-x)-400x(y-x^2),\,200(y-x^2)).
\]

在起点 (-1,1)：梯度 (-4,0)。以 eta=0.001 做一次参数更新：

\[
(x_1,y_1)=(-1,1)-0.001(-4,0)=(-0.996,1).
\]

下一次计算梯度时

\[
x_1^2=0.992016,\quad
y_1-x_1^2=0.007984,\quad
R_y(x_1,y_1)=200(0.007984)=1.5968.
\]

**谷地解释：** 对固定 x，y=x² 使第二个平方项为零，是 y 截面上的低谷，但 GD 在 (-1,1) 处的负梯度方向水平向右，更新 y 不变，导致新的点已经不满足 y=x²。如果想严格沿 y=x² 保持更新后位置，y 应变为 0.992016 而不是 1。于是下一步梯度的 y 分量变正，标准 GD 会减小 y。

教师实际 Python float64 计算第二步完整梯度与参数：

- grad0=(-4.0,0.0)
- theta1=(-0.996,1.0)
- grad1=(-0.8111744000000036,1.5967999999999982)
- theta2=(-0.9951888256,0.9984032)
- Loss0=4.0；Loss1=3.9903904256；Loss2=3.987182292617209。

需要区分 y=x² 是固定 x 下的 y 最优轨迹，不等于整个二维空间的驻点集合，也不应把它无条件视为几何法向距离意义下的精确谷底线。GD 的方向每步重新计算，与 Momentum 累积和 Adam 自适应机制形成后续联系。

In [ ]:
# 教师独立 Python 核验：两次标准 GD 更新与 Rosenbrock 解析梯度
import math

def rb_loss(p):
    x,y=p
    return (1-x)**2+100*(y-x*x)**2

def rb_grad(p):
    x,y=p
    return [-2*(1-x)-400*x*(y-x*x), 200*(y-x*x)]

eta=0.001
p0=[-1.0,1.0]
g0=rb_grad(p0)
p1=[p-eta*g for p,g in zip(p0,g0)]
g1=rb_grad(p1)
p2=[p-eta*g for p,g in zip(p1,g1)]
assert p1 == [-0.996,1.0]
assert math.isclose(g1[1],1.5968,rel_tol=1e-12)
assert math.isclose(p2[1],0.9984032,rel_tol=1e-12)
assert rb_loss(p0) > rb_loss(p1) > rb_loss(p2)
print("p1:",p1,"g1:",g1,"p2:",p2)
print("losses:",rb_loss(p0),rb_loss(p1),rb_loss(p2))
print("PASS: curvature-following dynamics and y gradient verified")

### 下一步理解检查：Rosenbrock 真正的驻点在哪里？（等待学生）

已知 Rosenbrock 的两个偏导：

\[
R_y(x,y)=200(y-x^2),\qquad
R_x(x,y)=-2(1-x)-400x(y-x^2).
\]

**请学生联立求解方程 R_x=R_y=0，求出所有二维驻点 (x,y)。** 提示：先利用 R_y=0 得到 y=x²，再代入另一个方程。最后结合 R(x,y)≥0 验证它是否是全局最小点。

这项补充不替代官方 Quiz；课程仍待损失地形高维二维切片解释、AdamW 解耦权重衰减、生产参数组与梯度裁剪、Ship It 和 stage=post 后测。

## Rosenbrock：求驻点与二阶判据——学生原始回答（2026-11-10）

> 是(1,1) 它的hessian是正定矩阵所以是全局最小点

### 验收：驻点与 Hessian 正定均正确，“因此全局最优”逻辑不成立

Rosenbrock 函数：

\[
R(x,y)=(1-x)^2+100(y-x^2)^2.
\]

梯度等于零的联立方程：

\[
\partial_yR=200(y-x^2)=0\Rightarrow y=x^2;
\quad
\partial_xR=-2(1-x)-400x(y-x^2)=0\Rightarrow x=1.
\]

因此唯一驻点是 (1,1)。在这个点，

\[
H_R(1,1)=\begin{pmatrix}802&-400\\-400&200\end{pmatrix},
\quad 802>0,\quad\det H=400>0.
\]

根据 2×2 对称矩阵的 Sylvester 判据，H 正定。加上 \(\nabla R(1,1)=0\)，**只能直接使用二阶充分条件推出 (1,1) 为严格局部最小点**，不是全局最小点。这是本次学生理解需要修正的地方。

**全局最优应另行证明**：对所有实数 x,y，两项平方均非负，所以 R(x,y)≥0；(1,1) 时 R=0，说明达到全局最小值 0；两个平方项同时为零仅有 x=1,y=1，因此全局解唯一。

**重要反例说明“局部 Hessian 正定不能保证全局最优”：**
取一元非凸函数 \(h(x)=x^2-3x^4+x^6\)，有 h'(0)=0、h''(0)=2>0，所以 x=0 是严格局部最小点；但 h(1)=1-3+1=-1<h(0)=0，故它不是全局最小点。这个反例从逻辑上否定了把单点二阶条件误当全局最优定理的推论。

后续学习重点是参数空间损失地形曲率与条件数，不重复已完成的官方 Exercises。

In [ ]:
# 教师独立复核：Rosenbrock 在 (1,1) 的 Hessian 特征值与条件数
import math
import numpy as np

def rb_hessian(x, y):
    return np.array([[2-400*y+1200*x*x, -400*x],
                     [-400*x, 200.]],dtype=float)

H=rb_hessian(1.0,1.0)
eigs=np.linalg.eigvalsh(H)
assert np.all(eigs>0)
assert np.linalg.det(H)>0
assert math.isclose(rosenbrock([1.0,1.0]),0,abs_tol=1e-12)
print("Hessian:",H)
print("Eigenvalues:",eigs)
print("Condition number:",eigs.max()/eigs.min())
# 在 x=0 处 h(x)=x^2-3x^4+x^6 的二阶导数为 2，且 h(1)=-1<h(0)=0。
print("Counterexample h(0),h(1):",0.0,1-3+1)
print("PASS: strict local vs global minima differentiated")

### 下一步：损失地形曲率与梯度下降步长的联系

H_R(1,1) 的两个正特征值，经教师实际 NumPy 计算近似为 0.399360767 和 1001.600639233。二范数条件数约 2508；这意味着附近两个主方向的二阶曲率差别非常大，是 Rosenbrock 狭窄谷地上 GD 可能遇到学习率权衡的一个直接证据（仅为局部近似）。

局部 Taylor 二阶展开：
\[
R(\theta^*+\delta)\approx R(\theta^*)+\tfrac12\delta^\top H\delta,
\]
因为在最优点 \(\nabla R(\theta^*)=0\)。

在 Hessian 的特征向量方向 \(v_i\) 上，令 \(\delta=av_i\)（取单位向量），上式约为 \(\frac12\lambda_i a^2\)。大曲率方向学习率太大可能震荡，小曲率方向需要更多步才能移动。

**下一道理解检查**：给定两个特征值 \(\lambda_{\mathrm{small}}\approx0.39936\)、\(\lambda_{\mathrm{large}}\approx1001.60064\)，若对一个局部二次近似使用固定 GD 步长 \(\eta=0.003\)，请判断大曲率方向的一步乘子 \(1-\eta\lambda_{\mathrm{large}}\) 的绝对值是否小于 1；据此说明该方向的误差会收缩还是放大。不要提前给答案。

## Rosenbrock Hessian 特征方向的固定 GD 稳定性理解题（2026-11-10）

**学生原始回答：**
> \(1-\eta\lambda\) = -2.00048 经过GD更新是增大了

### 验收：误差放大结论正确，乘法计算有误

题目指定的近似特征值 \(\lambda=1001.6\)、学习率 \(\eta=0.003\)，因此

\[
1-\eta\lambda=1-0.003(1001.6)=1-3.0048=\boxed{-2.0048}.
\]

学生写 -2.00048，是小数位计算错误；对 \(\lvert1-\eta\lambda\rvert>1\) 导致误差放大的判断正确。

**推导及条件**：对局部二次近似 \(q(\delta)=\frac12\delta^\top H\delta\) 取 Hessian 正特征值 \(\lambda>0\) 的特征向量，GD 误差更新为 \(e_{t+1}=e_t-\eta\lambda e_t=(1-\eta\lambda)e_t\)。因此：
- \(-1<1-\eta\lambda<1\) 等价于 \(0<\eta\lambda<2\)，该方向误差几何收缩；
- \(0<1-\eta\lambda<1\)：不跨越最优点的收缩；
- \(-1<1-\eta\lambda<0\)：跨越最优点但逐步收缩的振荡；
- \(1-\eta\lambda<-1\)：来回穿越且误差幅度增长。
- 在 \(\eta\lambda=2\) 时，倍数为 -1，误差幅度不减小；当 \(\eta\lambda=1\) 时理想二次方向一次变为零。

对给定 Hessian \(H_R(1,1)=\begin{pmatrix}802&-400\\-400&200\end{pmatrix}\)，教师实际使用 JS double 浮点核验其特征值：
\[
\lambda_{\max}\approx1001.6006392325123,\quad
\lambda_{\min}\approx0.39936076748762.
\]
本地二次模型的恒定 GD 步长收缩要求 \(0<\eta<2/\lambda_{\max}\approx0.0019968038374381654\)。用 \(\eta=0.003\) 超过这个界限；在 \(\eta=0.001\) 时大曲率方向因子为 -0.001600639232512302，小曲率方向却仍为 0.9996006392325124（推进非常缓慢）。

以上为**最优点局部二次模型的结论**，不意味着从任意远处出发时原始非二次 Rosenbrock 函数每一步误差都严格服从固定倍数公式，也不对 Momentum 或 Adam 自动适用。

In [ ]:
# 教师可复核：Rosenbrock 最优点处 Hessian 特征值与 GD 局部误差因子
import math
import numpy as np
H = np.array([[802.,-400.],[-400.,200.]])
eig = np.linalg.eigvalsh(H)
lam_min,lam_max = eig
assert np.all(eig > 0)
print("H eigenvalues:",eig)
print("eta stability upper bound:",2/lam_max)
for eta in (0.003,0.001,0.0005):
    factor_large = 1 - eta*lam_max
    factor_small = 1 - eta*lam_min
    print("eta",eta,"large curvature factor",factor_large,"small curvature factor",factor_small)
assert math.isclose(1-0.003*1001.6,-2.0048,rel_tol=1e-13)
assert abs(1-0.003*lam_max)>1
assert abs(1-0.001*lam_max)<1
print("PASS: spectral GD contraction checks")

### 下一道理解检查（等待学生）

保持最优点附近的 Rosenbrock 局部二次模型，特征值取 \(\lambda_{\max}\approx1001.6\)、\(\lambda_{\min}\approx0.39936\)，现在将学习率降至 \(\eta=0.001\)。

请学生独立计算 **两个**方向的更新因子：

\[
a_{\max}=1-0.001\lambda_{\max},\qquad
a_{\min}=1-0.001\lambda_{\min}.
\]

两个方向的误差都会收缩吗？如果收缩，为什么学习率已经足够稳定，GD 仍可能沿 Rosenbrock 狭窄谷地进展缓慢？

学生通过后可完成损失地形与局部曲率学习检查，进入尚未讲透的 AdamW 解耦 Weight Decay、生产训练 parameter groups 和梯度裁剪等 Use It 机制，之后 Ship It 和正式后测。根目录 LEARNING.md 未标记 01/08 完成。

## Hessian 谱方向 GD 收敛速度：学生原始回答（2026-11-10）

> a_max = -0.0016 a_min = 0.99960064 因为收缩的太慢了,a_min 太接近1了

### 验收：全部正确，局部曲率与损失地形学习检查通过

针对题目中的 \(\eta=0.001\)、\(\lambda_{\max}=1001.6\)、\(\lambda_{\min}=0.39936\)，学生正确计算出
\[
a_{\max}=1-0.001(1001.6)=-0.0016,\qquad
a_{\min}=1-0.001(0.39936)=0.99960064.
\]

两个因子的绝对值均小于 1，所以在此局部二次模型内误差收缩；但慢方向每一步仍然留下 99.960064% 的误差。经过 n 步误差的相对幅度为 \(|a_{\min}|^n\)，教师使用双精度实际计算：
- n=100：0.960843267404788
- n=1000：0.6706956877040995
- n=3000：0.3017008557703878
- n=5000：0.1357149122043296
- n=10000：0.018418537394628894

大曲率方向 \(|a_{\max}|=0.0016\) 的误差快速压低，而小曲率方向近似停滞，因此固定 GD 的整体局部线性收敛由慢方向控制。条件数约为 2508。此结论适用于 Rosenbrock 最优点附近的二次近似；不应冒充整个非线性训练轨迹的实测指标。

**课程覆盖**：Convex vs Non-convex 与 Loss Landscape（含实际 Rosenbrock 2D 等高线、Hessian、局部曲率、二维切片限制）已经完成学生理解检查。接下来进入 AdamW decoupled weight decay 与生产训练技术的补讲，不提前把整课标为完成。

**展示校正**：若绘制局部模型下 n=3000/5000 的误差比例，应分别使用 0.3017008557703878 / 0.1357149122043296；不要从图表中错误舍入出其他值。

## 官方 Use It 补课：AdamW 中的 decoupled weight decay

### 要解决的问题

在训练深度模型时，常希望模型权重不要无约束膨胀。传统 L2 惩罚在数据损失上叠加
\[
L_{\mathrm{reg}}(\theta)=L_{\mathrm{data}}(\theta)+\frac{\lambda}{2}\|\theta\|_2^2.
\]

导数 \(g_{\mathrm{reg}}=\nabla L_{\mathrm{data}}+\lambda\theta\)。在普通 GD 中，代入可得：
\[
\theta_{t+1}=\theta_t-\eta(g_{\mathrm{data}}+\lambda\theta_t)
=(1-\eta\lambda)\theta_t-\eta g_{\mathrm{data}}.
\]
所以普通 GD 中 L2 惩罚等价于乘性 weight decay（如果二者的 lambda 标度一致）。

### 为什么 Adam 中不是同一件事？

Adam 有历史的一阶矩 m 和二阶矩 v，并使用按参数缩放的更新：
\[
\Delta\theta=-\eta\frac{\widehat m}{\sqrt{\widehat v}+\epsilon}.
\]

若把 \(\lambda\theta\) 直接加进梯度，该项也进入 Adam 的 m、v，被自适应归一化处理，并不是纯粹的 \(-\eta\lambda\theta\) 收缩。这常被称为 **coupled L2**。

AdamW 则用真实数据损失梯度更新 m、v，将权重衰减从自适应梯度路径中解耦：
\[
\boxed{\theta_{t+1}=(1-\eta\lambda)\theta_t-\eta\frac{\widehat m_t}{\sqrt{\widehat v_t}+\epsilon}.}
\]

这里 \(\eta\) 为当前 LR、\(\lambda\) 为 weight_decay；每步纯权重衰减因子是 \(1-\eta\lambda\)，不是 \(1-\lambda\)。这个公式以两项均基于更新前 \(\theta_t\) 的值来表达单次更新。

### 一个可复核的具体例子

令单一权重 \(w=2\)、数据损失梯度为 0，\(\eta=0.01,\lambda=0.1\)，使用默认 \(\beta_1=0.9,\beta_2=0.999,\epsilon=10^{-8}\)。

- Adam 的默认 coupled L2 写法：正则梯度 \(g_{\mathrm{reg}}=0.1\times2=0.2\)；第一步 \(\widehat m=0.2,\widehat v=0.04\)，因此 \(w_1\approx2-0.01\times0.2/(0.2+10^{-8})\approx1.99\)。
- AdamW：数据梯度为 0，自适应更新项为 0，只应用单独的权重衰减：\(w_1=2(1-0.01\times0.1)=1.998\)。

教师使用 PyTorch 2.10.0+cpu、float64、foreach=False 实际执行单步得到：
- Adam(weight_decay=0.1): w=1.9900000005
- AdamW(weight_decay=0.1): w=1.998

**实验口径注意**：真实 PyTorch 需要参数的 grad 为存在的零张量，才会执行这些优化器对该参数的更新；若 grad 为 None，该参数通常会被跳过，包括 weight decay。例子使用 \(loss=0\cdot w\) 产生值为零但存在的 .grad。

**工程意义**：AdamW 的正则化力度与 LR \(\eta\) 和 weight_decay \(\lambda\) 共同相关；不要将 weight_decay=0.1 理解为每次直接减少 10% 的权重。参数组能区分需要/不需要衰减的参数（例如常见实现会排除 bias、Norm 参数），这一点在下一节进一步解释。

In [ ]:
# 教师真实运行的 PyTorch 2.10.0 CPU 验证，float64
import torch
for Optimizer in (torch.optim.Adam, torch.optim.AdamW):
    w = torch.nn.Parameter(torch.tensor(2.0, dtype=torch.float64))
    optimizer = Optimizer([w], lr=0.01, weight_decay=0.1, foreach=False)
    optimizer.zero_grad()
    loss = 0.0 * w     # 数值为 0，但反向传播会生成一个真实的零梯度
    loss.backward()
    assert w.grad is not None and w.grad.item() == 0.0
    optimizer.step()
    print(Optimizer.__name__, w.item())
    if Optimizer is torch.optim.Adam:
        assert abs(w.item()-1.9900000005)<1e-9
    else:
        assert abs(w.item()-1.998)<1e-12
print("PASS: coupled L2 in Adam and decoupled weight decay in AdamW differ")

### 下一道理解检查（等待学生）

针对 AdamW 公式，仍设 w=2、LR=0.01、weight_decay=0.1、数据梯度为零。

**为什么 weight_decay=0.1 不代表参数每一步直接缩小 10%？本例实际的每步权重衰减比例是多少？**

先让学生用公式解释再继续 PyTorch parameter groups、gradient clipping 和 Ship It。